
### A satellite-radar investigation of maritime behaviour, September 2025 to June 2026

**MSc Data Science, DSM050 Data Visualisation, Final Coursework**

---

The Automatic Identification System (AIS) is a cooperative source: a vessel that disables its
transponder produces no anomaly in the data, it ceases to appear. Synthetic aperture radar (SAR)
detects a hull from its reflected energy, by day or night and through cloud, irrespective of
whether the vessel is broadcasting. Comparing the two sources therefore measures which vessels
were present but not identifying themselves.

The radar also yields a length estimate for every detection, matched or not. That attribute
allows the same comparison to be made separately for each hull-length class, from coastal craft
to the largest vessels, and it is the basis of the study's principal finding.

## Research questions

**RQ1.** Did the share of vessels detected without a corresponding AIS broadcast change in the
Gulf, and is any change specific to the region rather than global?

**RQ2.** Does that change survive correction for satellite observation coverage and for the
mid-series change in the data provider's processing pipeline?

**RQ3.** How did the volume, spatial pattern and behaviour of traffic through the strait change,
and did vessels accumulate in place rather than transit?

**RQ4.** Does the picture change with vessel size? Did large ships leave the Gulf, or remain and
cease identifying themselves?

**RQ5.** Where did the identifiable traffic go? Which vessels left, and which ports and corridors
absorbed them?

## Structure and conventions

Sections 1 to 3 prepare the data. Section 4 introduces the world fleet by hull-length class,
Section 5 tests whether the Gulf change is measurable, Section 6 maps the strait for all vessels,
Section 7 repeats every map by size class, and Section 8 follows the identifiable fleet. Two
conventions apply throughout:

- every comparison is made per observed day rather than per calendar month, because two of the
  source months are truncated exports;
- every map is presented as a matched pre/post pair over windows containing the same number of
  observed days.

A full run takes about five minutes. `data/` holds the three prepared files;
`data/DATA_PROVENANCE.md` documents how they were derived.

---
# 1. Setup

In [1]:
import itertools, math, re, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import shapely
from shapely import box as shapely_box

warnings.filterwarnings("ignore", category=FutureWarning)
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print(f"pandas {pd.__version__} | numpy {np.__version__} | shapely {shapely.__version__}")

pandas 3.0.3 | numpy 2.4.4 | shapely 2.1.2


In [2]:
# ------------------------------------------------------------------ paths
PROJECT_DIR = Path.cwd()
DATA_DIR    = PROJECT_DIR / "data"
FIG_DIR     = PROJECT_DIR / "figures"
TAB_DIR     = PROJECT_DIR / "tables"
for _d in (FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)

AOI_CSV        = DATA_DIR / "hormuz_aoi_detections.csv.gz"
WORLD_CSV      = DATA_DIR / "world_detections_slim.csv.gz"
FOOTPRINTS_CSV = DATA_DIR / "scene_footprints_aoi.csv.gz"
for _f in (AOI_CSV, WORLD_CSV, FOOTPRINTS_CSV):
    if not _f.exists():
        raise FileNotFoundError(f"Missing {_f}. Run this notebook from DSM050_Final_v2/.")

# ------------------------------------------------------------ study areas
# Bounding boxes are (lon_min, lat_min, lon_max, lat_max).
AOI = (30.0, 0.0, 78.0, 32.0)          # Red Sea -> Arabian Sea -> west India

REGIONS = {
    "strait_of_hormuz": (54.0, 24.0, 58.5, 27.5),
    "persian_gulf":     (47.0, 23.0, 56.5, 31.0),
    "gulf_of_oman":     (56.5, 22.0, 62.0, 27.0),
    "arabian_sea":      (58.0, 8.0,  72.0, 25.0),
    "gulf_of_aden":     (43.0, 10.0, 52.0, 15.5),
    "red_sea":          (32.0, 12.0, 43.5, 30.0),
    "bab_el_mandeb":    (42.5, 11.5, 44.5, 13.5),
}

# Narrow gate across the strait used for transit counting.
HORMUZ_GATE = (55.6, 25.4, 57.4, 27.0)

# ------------------------------------------------------------------ ports
# (lon, lat) of the principal ports/terminals relevant to Gulf routing.
PORTS = {
    "Jebel Ali (AE)":        (55.027, 25.010),
    "Khalifa/Abu Dhabi (AE)":(54.650, 24.807),
    "Fujairah (AE)":         (56.359, 25.166),
    "Ruwais (AE)":           (52.730, 24.140),
    "Ras Tanura (SA)":       (50.160, 26.700),
    "Jubail (SA)":           (49.660, 27.030),
    "Yanbu (SA)":            (38.060, 24.090),
    "Jeddah (SA)":           (39.150, 21.480),
    "Kharg Island (IR)":     (50.320, 29.230),
    "Bandar Abbas (IR)":     (56.210, 27.150),
    "Umm Qasr (IQ)":         (47.930, 30.030),
    "Basra Oil Terminal(IQ)":(48.810, 29.690),
    "Kuwait/Al Ahmadi (KW)": (48.150, 29.070),
    "Hamad (QA)":            (51.600, 25.010),
    "Ras Laffan (QA)":       (51.540, 25.910),
    "Mina Salman (BH)":      (50.610, 26.200),
    "Sohar (OM)":            (56.620, 24.500),
    "Duqm (OM)":             (57.700, 19.670),
    "Salalah (OM)":          (54.000, 16.940),
    "Aden (YE)":             (45.020, 12.790),
    "Djibouti (DJ)":         (43.140, 11.600),
    "Karachi (PK)":          (66.980, 24.840),
    "Mundra (IN)":           (69.720, 22.740),
    "Jawaharlal Nehru (IN)": (72.950, 18.950),
}

# ------------------------------------------------------------- data model
# The pipeline assigns matched_category='unmatched' whenever matching_score
# falls below this value.  Verified empirically: every named category has
# matching_score >= 0.01, every 'unmatched' row has matching_score < 0.01.
MATCH_SCORE_THRESHOLD = 0.01

# Three-tier AIS-correlation taxonomy (see hormuz.build.add_ais_status).
AIS_STATUS = ["matched", "weak_match", "no_ais_candidate"]

VESSEL_CATEGORIES = [
    "cargo", "other", "fishing", "passenger", "gear",
    "seismic_vessel", "carrier", "bunker", "noisy_vessel", "discrepancy",
]

# The GFW detection pipeline changes version inside the study window.
# Any count compared across this boundary is confounded by the version change.
PIPELINE_BREAK = "2026-03"

# ---------------------------------------------------------------- plotting
SURFACE = "#0b1420"          # chart/ocean surface; validator runs against this
INK      = "#e1e4ea"
INK_MUTED = "#8f98a8"
GRID     = "#1c2836"

MAP_THEME = dict(
    land="#232830", ocean=SURFACE, country="#4c5563",
    coastline="#6b7686", bg=SURFACE, font=INK, grid=GRID,
)

# Categorical slots 1/2/3 of the reference palette, dark steps.  Verified with
# hormuz.palette_check.validate(..., mode="dark", surface=SURFACE, pairs="all"):
# lightness band PASS, chroma PASS, CVD all-pairs worst 9.4 (deutan), normal
# vision worst 20.9, contrast all >= 3:1.  The intuitive red-for-dark palettes
# were rejected - orange/red scores Delta E 7.1 for normal vision, below the 15
# floor, so red is not available alongside orange here.
CAT_COLOURS = {
    "matched":          "#3987e5",   # slot 1, blue
    "no_ais_candidate": "#d95926",   # slot 2, orange - the study's focus series
    "weak_match":       "#199e70",   # slot 3, aqua
}
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIVERGING = ("#3987e5", "#383835", "#e66767")   # blue <-> gray <-> red

# --------------------------------------------------- map colour scales
# SEQ_BLUE runs light-to-dark, which is correct on a white page and wrong on
# this one.  Used directly on the dark ocean it inverts the reading: empty water
# lands on the palest step at ~14:1 contrast and dominates, while the busiest cells
# land on the darkest step at ~1.6:1 and are lost against the sea.
#
# SEQ_BLUE_DARK keeps the single blue hue but runs it the other way for a dark
# surface: deep navy at zero, sitting just above the ocean colour so empty water
# recedes, climbing to near-white for the busiest cells.  Both lightness
# (OKLCH L 0.26 -> 0.96) and contrast (1.2:1 -> 16.4:1) increase monotonically,
# so the brightest mark is always the busiest water.  The steps are the
# documented blue ramp, read dark-to-light.
SEQ_BLUE_DARK = [
    (0.00, "#0d2440"),   # ~ocean: nothing here
    (0.15, "#104281"),
    (0.32, "#1c5cab"),
    (0.50, "#2a78d6"),
    (0.68, "#5598e7"),
    (0.84, "#9ec5f4"),
    (1.00, "#eaf2fe"),   # brightest = busiest
]

# Warm alternative, kept for reference: same direction, hotter top end.
SEQ_HEAT = [
    (0.00, "#16233a"), (0.20, "#4a1f52"), (0.40, "#8f2742"),
    (0.60, "#c9432a"), (0.80, "#e8801f"), (1.00, "#f5cf4e"),
]

# Diverging scale for signed change: blue below zero, warm above it, with a
# dark neutral midpoint so that "no change" recedes.  The warm arm runs
# yellow -> orange -> deep red; red is slightly darker than yellow, so on this
# arm magnitude is carried by hue and chroma (C 0.16 -> 0.19) rather than by
# lightness.
DIV_BLUE_YELLOW_RED = [
    (0.00, "#5fa8f0"),   # largest decrease
    (0.25, "#2f6fb5"),
    (0.50, "#2b3240"),   # no change - recedes into the map
    (0.72, "#eda100"),
    (0.86, "#e8791f"),
    (1.00, "#c0392b"),   # largest increase
]

# ------------------------------------------------------- vessel groupings
# GFW's raw categories are too unbalanced to colour directly: in the Hormuz
# segments 'other' holds 8,177 rows and 'carrier' holds 5.  These four groups
# keep every class with real support separate and fold the long tail into one
# deliberately recessive grey slot.
VESSEL_GROUPS = {
    "cargo": "Cargo",
    "other": "Unclassified ('other')",
    "fishing": "Fishing / gear",
    "gear": "Fishing / gear",
    "passenger": "Specialised / other",
    "seismic_vessel": "Specialised / other",
    "bunker": "Specialised / other",
    "carrier": "Specialised / other",
    "noisy_vessel": "Specialised / other",
    "discrepancy": "Specialised / other",
}
VESSEL_GROUP_ORDER = ["Cargo", "Unclassified ('other')", "Fishing / gear",
                      "Specialised / other"]
# Categorical slots 1/2/3 again (validated all-pairs on this surface), plus a
# neutral grey for the pooled tail.  Grey is not a categorical hue - it is the
# recessive slot, and carries no identity claim.
VESSEL_GROUP_COLOURS = {
    "Cargo":                  "#3987e5",
    "Unclassified ('other')": "#d95926",
    "Fishing / gear":         "#199e70",
    "Specialised / other":    "#6b7280",
}

# ------------------------------------------------- comparison time windows
# POST covers four months (the data ends 2026-06-29), so the pre-conflict
# window is the four months immediately before the change point rather than all
# six available.  Comparing 6 months against 4 makes the later panel look
# emptier for purely arithmetic reasons.
PRE_FULL    = ["2025-09", "2025-10", "2025-11", "2025-12", "2026-01", "2026-02"]
PRE_MATCHED = ["2025-11", "2025-12", "2026-01", "2026-02"]
POST_WINDOW = ["2026-03", "2026-04", "2026-05", "2026-06"]

# ------------------------------------------------------- wider-scale gates
# Chokepoints and the routes that bypass them.  Boxes are
# (lon_min, lat_min, lon_max, lat_max).
CORRIDORS = {
    "Strait of Hormuz":    (55.6, 25.4, 57.4, 27.0),
    "Bab el-Mandeb":       (42.5, 11.5, 44.5, 13.5),
    "Suez / Gulf of Suez": (32.0, 27.0, 34.5, 30.2),
    "Cape of Good Hope":   (15.0, -38.0, 31.0, -33.0),
    "Mozambique Channel":  (35.0, -26.0, 46.0, -12.0),
    "Strait of Malacca":   (98.0, 1.0, 105.0, 6.5),
    "Gulf of Guinea":      (-5.0, -2.0, 9.0, 6.0),
}
# Which corridors are alternatives to each other, for the re-routing narrative.
ROUTE_PAIRS = {
    "Suez vs Cape": ("Suez / Gulf of Suez", "Cape of Good Hope"),
    "Bab el-Mandeb vs Cape": ("Bab el-Mandeb", "Cape of Good Hope"),
}

# ---------------------------------------------------- analysis shorthands
GULF_REGIONS = ["strait_of_hormuz", "persian_gulf", "gulf_of_oman"]
GULF_BOX     = (47.0, 22.0, 62.0, 31.0)
HORMUZ_BOX   = (50.0, 22.0, 62.0, 30.5)
WORLD_BOX    = (-30.0, -45.0, 130.0, 62.0)
WINDOWS = [("Nov 2025 - Feb 2026", PRE_MATCHED, "pre"),
           ("Mar - Jun 2026", POST_WINDOW, "post")]

# Point budgets for the copies embedded in this notebook; full-resolution
# interactive versions are written to figures/.
MAP_MAX_POINTS   = 25_000
ROUTE_MAX_SEGS   = 9_000
WORLD_MAX_POINTS = 30_000
WORLD_ROUTE_SEGS = 15_000

# The analysis functions were developed as a package and call C.X, viz.X and so
# on; binding those names to this notebook's own module resolves every such call
# to the definitions in these cells, unchanged.
_self = sys.modules["__main__"]
C = build = cvg = coverage = analysis = tracks = ports = viz = _self
wider = vessels = _self

print(f"data    : {DATA_DIR}")
print(f"figures : {FIG_DIR}")

data    : c:\Users\tatee\Desktop\Data Vis\CW2\DSM050_Final_v2\data
figures : c:\Users\tatee\Desktop\Data Vis\CW2\DSM050_Final_v2\figures


---
# 2. Data and preparation

**Source.** Sentinel-1 SAR vessel detections published by Global Fishing Watch (GFW), covering
1 September 2025 to 29 June 2026. Each record is one radar return classified as a vessel, with
position, timestamp, a length estimated from the radar signature, three confidence scores, and,
where the detection correlated to a concurrent AIS broadcast, an MMSI and vessel class. A
companion product gives each scene's water-masked footprint.

The imagery is from ESA's Sentinel-1 mission under the Copernicus open data policy; GFW's
detection method is peer-reviewed (Paolo et al., 2024). The licence permits non-commercial
research with attribution. An MMSI identifies a hull, not a person. The substantive ethical
consideration is dual use, so the study works with aggregates over a completed period rather
than tracking named vessels in near real time.

**Three defects handled during preparation.** The June export is duplicated (one file covers
1 to 27 June, another 1 to 29 June, and a wildcard read counts June twice). February 2026 ends on
the 25th and May on the 28th, so calendar-month counts fall for clerical reasons. The
`matched_category` label is ambiguous, which is addressed in Section 2.1.

In [3]:
TS_FORMAT = "%Y-%m-%d %H:%M:%S UTC"

def read_detections(path, world=False):
    """Read a packaged CSV and apply compact dtypes.

    With pandas defaults every score becomes a float64, every MMSI a float64 and
    every 67-character scene id a separate Python string. Typing each column to
    the range it occupies cuts the study-area table from 55 MB to 10 MB.

    The MMSI case is not only about size: read as a float, 367093000 becomes
    367093000.0, and any later string join silently keys on a different value.
    """
    raw = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    out = pd.DataFrame(index=raw.index)

    # Source files stamp times as "... UTC"; the prepared files carry no suffix.
    ts = pd.to_datetime(raw["timestamp"], format=TS_FORMAT, errors="coerce")
    if ts.isna().mean() > 0.5:
        ts = pd.to_datetime(raw["timestamp"], errors="coerce")
    if ts.isna().all():
        raise ValueError(f"Could not parse any timestamp in {path.name}")
    out["timestamp"] = ts.astype("datetime64[s]")

    for col in ["lat", "lon", "presence_score", "length_m",
                "matching_score", "fishing_score"]:
        if col in raw:
            out[col] = pd.to_numeric(raw[col], errors="coerce").astype("float32")

    out["mmsi"] = pd.to_numeric(raw["mmsi"], errors="coerce").astype("UInt32")
    cat = raw["matched_category"]
    out["matched_category"] = cat.where(cat != "", other=None).astype("category")
    if "scene_id" in raw and not world:
        out["scene_id"] = raw["scene_id"].astype("category")

    out.attrs["naive_bytes"] = raw.memory_usage(deep=True).sum()
    return out.sort_values("timestamp", ignore_index=True)


def memory_report(df):
    usage = df.memory_usage(deep=True).drop("Index", errors="ignore")
    return (pd.DataFrame({"bytes": usage,
                          "dtype": [str(df[c].dtype) for c in usage.index]})
            .assign(MB=lambda d: d["bytes"] / 1e6)
            .sort_values("bytes", ascending=False))


def in_box(df, box):
    lon_min, lat_min, lon_max, lat_max = box
    return df["lon"].between(lon_min, lon_max) & df["lat"].between(lat_min, lat_max)


def label_regions(df):
    region = pd.Series(pd.NA, index=df.index, dtype=object)
    for name, box in REGIONS.items():
        region[in_box(df, box) & region.isna()] = name
    df["region"] = region.astype("category")
    return df


def period_of(df):
    return df["timestamp"].dt.to_period("M").astype(str)

## 2.1 Ambiguity in the `matched_category` label

The column appears to record whether a vessel was broadcasting, and does not. Cross-tabulated
against the continuous `matching_score`, the label `unmatched` covers two situations:

- **no AIS candidate**: `mmsi` blank, `matching_score` exactly zero;
- **a rejected weak candidate**: `mmsi` populated, `0 < matching_score < 0.01`.

Every named class scores at least 0.01, which establishes that value as the acceptance threshold
the label encodes. Conflating the two overstates dark activity by about a third, so the study
uses a three-tier status and reports the middle tier separately.

In [4]:
def add_ais_status(df):
    """Three-tier AIS-correlation status (see discussion above)."""
    score = df["matching_score"].fillna(0).to_numpy(dtype="float32")
    has_mmsi = df["mmsi"].notna().to_numpy()
    status = np.where(score >= MATCH_SCORE_THRESHOLD, "matched",
                      np.where(has_mmsi | (score > 0), "weak_match",
                               "no_ais_candidate"))
    df["ais_status"] = pd.Categorical(status, categories=AIS_STATUS)
    return df


t0 = time.time()
det      = label_regions(add_ais_status(read_detections(AOI_CSV)))
glob_det = add_ais_status(read_detections(WORLD_CSV, world=True))
gulf     = det[det["region"].isin(GULF_REGIONS)]

print(f"AOI detections   : {len(det):,}")
print(f"Gulf detections  : {len(gulf):,}")
print(f"World detections : {len(glob_det):,}")
print(f"Date range       : {det.timestamp.min()} to {det.timestamp.max()}")
print(f"loaded in {time.time()-t0:.1f}s")

AOI detections   : 239,712
Gulf detections  : 136,717
World detections : 2,394,698
Date range       : 2025-09-01 00:41:22 to 2026-06-29 15:28:06
loaded in 26.6s


In [5]:
rep = memory_report(det)
naive, compact = det.attrs["naive_bytes"] / 1e6, rep.MB.sum()
print(f"AOI table as read : {naive:6.1f} MB")
print(f"after typing      : {compact:6.1f} MB   ({naive/compact:.1f}x smaller)")

status = det.groupby("ais_status", observed=True).agg(
    detections=("lat", "size"),
    with_mmsi=("mmsi", lambda s: s.notna().sum()),
    min_score=("matching_score", "min"),
    max_score=("matching_score", "max"),
    median_length_m=("length_m", "median"))
status["share_pct"] = (100 * status["detections"] / len(det)).round(1)
display(status.round(3))

raw_un = det[det.matched_category == "unmatched"]
print(f"rows labelled 'unmatched'      : {len(raw_un):,}")
print(f"  genuinely no AIS candidate   : {raw_un.mmsi.isna().sum():,}")
print(f"  carrying a rejected MMSI     : {raw_un.mmsi.notna().sum():,}")
print(f"  overstatement if conflated   : "
      f"{100*raw_un.mmsi.notna().sum()/raw_un.mmsi.isna().sum():.0f}%")

AOI table as read :   55.1 MB
after typing      :   10.3 MB   (5.3x smaller)


,detections,with_mmsi,min_score,max_score,median_length_m,share_pct
ais_status,,,,,,
matched,122648,122648,0.01,1530.146973,122.385002,51.2
weak_match,30196,30196,0.00,0.010000,23.604000,12.6
no_ais_candidate,86868,0,0.00,0.000000,22.653999,36.2


rows labelled 'unmatched'      : 117,064
  genuinely no AIS candidate   : 86,868
  carrying a rejected MMSI     : 30,196
  overstatement if conflated   : 35%


---
# 3. Observation coverage

A fall in detections has three possible causes: fewer vessels, less sea imaged, or a change in
the provider's processing. The second is addressed here, the third in Section 5.

Each Sentinel-1 scene carries a water-masked footprint recording the sea it covered. Rasterising
those footprints onto a 0.1 degree grid gives a daily map of observed water, so that any statistic
can be expressed per unit of sea actually imaged.

In [6]:
import numpy as np
import pandas as pd
import shapely
from shapely import box as shapely_box


# Grid resolution in degrees. 0.1 deg is ~11 km lat, ~10 km lon at 25N - fine
# enough to separate the strait from the open Gulf, coarse enough that a
# whole-study raster stays in memory.
GRID_RES = 0.1

EARTH_R_KM = 6371.0088


# --------------------------------------------------------------- grid helpers
def grid_shape(res=GRID_RES, aoi=None):
    lon_min, lat_min, lon_max, lat_max = aoi or C.AOI
    nx = int(round((lon_max - lon_min) / res))
    ny = int(round((lat_max - lat_min) / res))
    return nx, ny


def cell_index(lon, lat, res=GRID_RES, aoi=None):
    """Vectorised (lon, lat) -> integer grid indices (ix, iy)."""
    lon_min, lat_min, lon_max, lat_max = aoi or C.AOI
    ix = np.floor((np.asarray(lon, dtype="float64") - lon_min) / res).astype("int32")
    iy = np.floor((np.asarray(lat, dtype="float64") - lat_min) / res).astype("int32")
    return ix, iy


def cell_centres(ix, iy, res=GRID_RES, aoi=None):
    lon_min, lat_min, lon_max, lat_max = aoi or C.AOI
    return (lon_min + (np.asarray(ix) + 0.5) * res,
            lat_min + (np.asarray(iy) + 0.5) * res)


def cell_area_km2(iy, res=GRID_RES, aoi=None):
    """Area of a grid cell at row ``iy``, accounting for meridian convergence."""
    lon_min, lat_min, lon_max, lat_max = aoi or C.AOI
    lat0 = lat_min + np.asarray(iy) * res
    lat1 = lat0 + res
    # Exact spherical band area, scaled by the cell's longitude fraction.
    band = (2 * np.pi * EARTH_R_KM ** 2
            * np.abs(np.sin(np.radians(lat1)) - np.sin(np.radians(lat0))))
    return band * (res / 360.0)


# ------------------------------------------------------------------ footprints
def load_footprints(path, aoi=None, chunksize=500, verbose=False):
    """Read one footprints CSV, returning AOI-clipped geometries.

    The WKT strings run to ~30 kB each, so the file is read in small chunks and
    everything outside the AOI is discarded before anything is retained.
    """
    aoi = aoi or C.AOI
    clip = shapely_box(*aoi)
    rows = []

    reader = pd.read_csv(
        path,
        usecols=["scene_id", "date", "start_time", "end_time", "footprint_wkt"],
        dtype=str, chunksize=chunksize,
    )
    for chunk in reader:
        geoms = shapely.from_wkt(chunk["footprint_wkt"].to_numpy())
        keep = shapely.intersects(geoms, clip)
        if not keep.any():
            continue
        sub = chunk.loc[keep, ["scene_id", "date", "start_time", "end_time"]].copy()
        sub["geometry"] = shapely.intersection(geoms[keep], clip)
        rows.append(sub)

    if not rows:
        return pd.DataFrame(columns=["scene_id", "date", "start_time",
                                     "end_time", "geometry"])
    out = pd.concat(rows, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"]).dt.date
    if verbose:
        print(f"  {path.name:46s} {len(out):>6,} AOI scenes", flush=True)
    return out


def rasterise(geoms, res=GRID_RES, aoi=None):
    """Grid cells whose centre falls inside any of ``geoms``.

    Returns a set-like array of unique (ix, iy) pairs.
    """
    aoi = aoi or C.AOI
    lon_min, lat_min, lon_max, lat_max = aoi
    nx, ny = grid_shape(res, aoi)

    hits = set()
    for geom in geoms:
        if geom is None or geom.is_empty:
            continue
        gx0, gy0, gx1, gy1 = geom.bounds
        ix0 = max(0, int(np.floor((gx0 - lon_min) / res)))
        ix1 = min(nx - 1, int(np.floor((gx1 - lon_min) / res)))
        iy0 = max(0, int(np.floor((gy0 - lat_min) / res)))
        iy1 = min(ny - 1, int(np.floor((gy1 - lat_min) / res)))
        if ix1 < ix0 or iy1 < iy0:
            continue

        gx, gy = np.meshgrid(np.arange(ix0, ix1 + 1), np.arange(iy0, iy1 + 1))
        gx = gx.ravel()
        gy = gy.ravel()
        cx = lon_min + (gx + 0.5) * res
        cy = lat_min + (gy + 0.5) * res
        inside = shapely.contains_xy(geom, cx, cy)
        if inside.any():
            hits.update(zip(gx[inside].tolist(), gy[inside].tolist()))
    return hits


def build_coverage(res=GRID_RES, verbose=True):
    """Rasterise every day's footprints into a long (date, ix, iy) table."""
    paths = sorted(C.DATA_DIR.glob(C.FOOTPRINTS_GLOB))
    if not paths:
        raise FileNotFoundError(f"No files matching {C.FOOTPRINTS_GLOB}")

    scene_rows = []
    cover_rows = []
    seen_scenes = set()

    for path in paths:
        fp = load_footprints(path, verbose=verbose)
        if fp.empty:
            continue
        # The June re-export repeats scenes already read; keep first occurrence.
        fresh = ~fp["scene_id"].isin(seen_scenes)
        fp = fp[fresh]
        seen_scenes.update(fp["scene_id"])
        if fp.empty:
            continue

        scene_rows.append(fp[["scene_id", "date", "start_time", "end_time"]])

        for day, grp in fp.groupby("date", sort=True):
            hits = rasterise(grp["geometry"].to_numpy(), res=res)
            if not hits:
                continue
            arr = np.fromiter((v for pair in hits for v in pair),
                              dtype="int32", count=2 * len(hits)).reshape(-1, 2)
            cover_rows.append(pd.DataFrame({
                "date": pd.Timestamp(day),
                "ix": arr[:, 0],
                "iy": arr[:, 1],
                "n_scenes": len(grp),
            }))

    scenes = pd.concat(scene_rows, ignore_index=True)
    scenes["date"] = pd.to_datetime(scenes["date"])
    scenes.to_parquet(C.PQ_SCENES, compression="zstd", index=False)

    cov = pd.concat(cover_rows, ignore_index=True)
    cov = cov.drop_duplicates(subset=["date", "ix", "iy"])
    cov["area_km2"] = cell_area_km2(cov["iy"].to_numpy(), res=res).astype("float32")
    cov["ix"] = cov["ix"].astype("int16")
    cov["iy"] = cov["iy"].astype("int16")
    cov.to_parquet(C.PQ_COVERAGE, compression="zstd", index=False)

    if verbose:
        print(f"\nCoverage grid: {len(cov):,} observed cell-days "
              f"over {cov['date'].nunique()} days, {len(scenes):,} AOI scenes")
    return cov


def load_coverage():
    if not C.PQ_COVERAGE.exists():
        raise FileNotFoundError(f"{C.PQ_COVERAGE} missing - run build_coverage()")
    return pd.read_parquet(C.PQ_COVERAGE)


# -------------------------------------------------------------- region tagging
def tag_cells_with_region(cov, res=GRID_RES, aoi=None):
    """Attach a REGIONS label to each grid cell by its centre point."""
    lon, lat = cell_centres(cov["ix"].to_numpy(), cov["iy"].to_numpy(), res, aoi)
    region = np.full(len(cov), "", dtype=object)
    for name, (lo0, la0, lo1, la1) in C.REGIONS.items():
        m = (region == "") & (lon >= lo0) & (lon <= lo1) & (lat >= la0) & (lat <= la1)
        region[m] = name
    out = cov.copy()
    out["region"] = pd.Categorical(region)
    return out


def daily_region_coverage(cov=None, res=GRID_RES):
    """Observed sea area per day per named region (km2)."""
    cov = cov if cov is not None else load_coverage()
    tagged = tag_cells_with_region(cov, res=res)
    tagged = tagged[tagged["region"] != ""]
    out = (tagged.groupby(["date", "region"], observed=True)["area_km2"]
           .sum().reset_index(name="observed_km2"))
    return out


def normalised_counts(det, cov=None, res=GRID_RES, freq="D"):
    """Detections per 1000 km2 of observed sea, by period and region.

    This is the headline correction the study rests on: it removes the
    satellite's acquisition schedule from every trend line.
    """
    cov = cov if cov is not None else load_coverage()
    obs = daily_region_coverage(cov, res=res)
    obs["period"] = obs["date"].dt.to_period(freq).astype(str)
    obs_agg = (obs.groupby(["period", "region"], observed=True)["observed_km2"]
               .sum().reset_index())

    d = det.dropna(subset=["region"]).copy()
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)
    det_agg = (d.groupby(["period", "region"], observed=True)
               .size().reset_index(name="detections"))

    merged = det_agg.merge(obs_agg, on=["period", "region"], how="outer").fillna(
        {"detections": 0, "observed_km2": 0.0})
    merged["per_1000km2"] = np.where(
        merged["observed_km2"] > 0,
        1000.0 * merged["detections"] / merged["observed_km2"],
        np.nan,
    )
    return merged.sort_values(["region", "period"], ignore_index=True)

In [7]:
def load_footprints_csv(path, aoi=None):
    """Read the prepared footprint file and clip to the study area."""
    aoi = aoi or AOI
    clip = shapely_box(*aoi)
    raw = pd.read_csv(path, dtype=str)
    # make_valid guards against self-intersections left by simplification.
    geoms = shapely.make_valid(shapely.from_wkt(raw["footprint_wkt"].to_numpy()))
    keep = shapely.intersects(geoms, clip)
    out = raw.loc[keep, ["scene_id", "date", "start_time", "end_time"]].copy()
    out["geometry"] = shapely.intersection(geoms[keep], clip)
    out["date"] = pd.to_datetime(out["date"]).dt.date
    return out


t0 = time.time()
fp = load_footprints_csv(FOOTPRINTS_CSV)
rows = []
for day, grp in fp.groupby("date", sort=True):
    hits = rasterise(grp["geometry"].to_numpy())
    if not hits:
        continue
    arr = np.array(sorted(hits), dtype="int32")
    rows.append(pd.DataFrame({"date": pd.Timestamp(day), "ix": arr[:, 0],
                              "iy": arr[:, 1], "n_scenes": len(grp)}))

cov = pd.concat(rows, ignore_index=True).drop_duplicates(subset=["date", "ix", "iy"])
cov["area_km2"] = cell_area_km2(cov["iy"].to_numpy()).astype("float32")

obs = daily_region_coverage(cov)
monthly_obs = (obs[obs.region.isin(GULF_REGIONS)]
               .assign(period=lambda d: d.date.dt.to_period("M").astype(str))
               .groupby("period", observed=True)["observed_km2"].sum())
print(f"{len(fp):,} AOI scenes -> {len(cov):,} observed cell-days ({time.time()-t0:.1f}s)")
print(f"busiest month / quietest = {monthly_obs.max()/monthly_obs.min():.2f}x "
      "- enough to manufacture a trend unaided")
display(monthly_obs.round(0).to_frame("sea area imaged (km2)"))

2,533 AOI scenes -> 398,739 observed cell-days (6.6s)
busiest month / quietest = 1.35x - enough to manufacture a trend unaided


,sea area imaged (km2)
period,
2025-12,1954933.0
2026-01,2228892.0
2026-02,1662727.0
2026-03,2241096.0
2026-04,1845351.0
2026-05,1938864.0
2026-06,1909952.0


Footprints are published only from December 2025, so coverage-corrected figures begin there and
September–November are compared on raw counts, labelled as such.

---
# 4. The world fleet and its ship types

Every count in this study can be separated by hull length, because radar measures length for
every detection, matched or not. This section defines the classes and describes the geography of
each.

## 4.1 Hull-length classes

Band edges follow the standard containership generations rather than round numbers, so that each
band corresponds to a real class of ship: feeder and handysize hulls below 200 m, the Panamax
generations between 200 and 300 m, and post-Panamax, New-Panamax and the ultra-large classes
above 300 m (Rodrigue, 2024).

![Containership classes](figures/fig01_containership_classes_rodrigue.png)

*Figure 1. Evolution of containership classes. Source: Rodrigue (2024).*

Radar-derived length varies with sea state and viewing aspect, so a hull near a boundary may fall
on either side of it. The bands are 100 m wide so that such blurring does not move the aggregate
pattern.

**The filter is on length, not vessel class.** `matched_category` is only meaningful for
detections that matched AIS; every dark detection carries the literal value `unmatched`. A class
filter would delete the entire dark population and make the central comparison impossible. Length
is present for every detection and is the only attribute that can filter both sides.

In [8]:
import numpy as np
import pandas as pd


# Regions treated as "the Gulf" for the treatment group.
TREATMENT_BOX = (47.0, 22.0, 62.0, 31.0)

CONTROL_BOXES = {
    "NE Atlantic":     (-15.0, 45.0, 5.0, 62.0),
    "SE Asia":         (100.0, -10.0, 125.0, 20.0),
    "Gulf of Mexico":  (-98.0, 18.0, -80.0, 31.0),
    "W Mediterranean": (-5.0, 35.0, 16.0, 45.0),
}


# ------------------------------------------------------------------- helpers
def add_period(df, freq="M"):
    out = df.copy()
    out["period"] = out["timestamp"].dt.to_period(freq).astype(str)
    return out


def status_table(df, freq="M"):
    """Counts and dark share per period."""
    d = add_period(df, freq)
    piv = (d.pivot_table(index="period", columns="ais_status", values="lat",
                         aggfunc="count", observed=True)
           .reindex(columns=C.AIS_STATUS).fillna(0.0))
    piv["total"] = piv.sum(axis=1)
    piv["dark_share"] = 100 * piv["no_ais_candidate"] / piv["total"].replace(0, np.nan)
    piv["matched_share"] = 100 * piv["matched"] / piv["total"].replace(0, np.nan)
    return piv.reset_index()


def dark_share_series(df, freq="M"):
    t = status_table(df, freq)
    return t.set_index("period")["dark_share"]


# ------------------------------------------------- difference-in-differences
def treatment_control_table(global_df, freq="M"):
    """Dark share for the Gulf and every control region, side by side."""
    cols = {}
    cols["World"] = dark_share_series(global_df, freq)
    for name, box in CONTROL_BOXES.items():
        cols[name] = dark_share_series(global_df[build.in_box(global_df, box)], freq)
    cols["Gulf (treatment)"] = dark_share_series(
        global_df[build.in_box(global_df, TREATMENT_BOX)], freq)
    return pd.DataFrame(cols)


def did_estimate(global_df, pre_periods, post_periods, freq="M"):
    """Difference-in-differences on dark share: Gulf minus world baseline.

    Returns a one-row frame with the pre/post means for treatment and control
    and the resulting DiD estimate in percentage points.
    """
    tab = treatment_control_table(global_df, freq)
    treat = tab["Gulf (treatment)"]
    ctrl = tab["World"]

    t_pre, t_post = treat.loc[pre_periods].mean(), treat.loc[post_periods].mean()
    c_pre, c_post = ctrl.loc[pre_periods].mean(), ctrl.loc[post_periods].mean()

    return pd.DataFrame([{
        "treat_pre": t_pre, "treat_post": t_post, "treat_delta": t_post - t_pre,
        "control_pre": c_pre, "control_post": c_post, "control_delta": c_post - c_pre,
        "did_pp": (t_post - t_pre) - (c_post - c_pre),
    }]).round(2)


def pipeline_break_check(global_df, freq="M", break_period=None):
    """How much each region moved across the GFW v3 -> v4 boundary.

    If the pipeline change were driving the result, every region would move
    together.
    """
    break_period = break_period or C.PIPELINE_BREAK
    tab = treatment_control_table(global_df, freq)
    periods = list(tab.index)
    if break_period not in periods:
        raise ValueError(f"{break_period} not in {periods}")
    i = periods.index(break_period)
    before, after = periods[i - 1], periods[i]
    delta = (tab.loc[after] - tab.loc[before]).rename(f"{before} -> {after} (pp)")
    return delta.to_frame().round(2)


# -------------------------------------------------------------- change point
def cusum_changepoint(series, min_size=3):
    """Single most likely change point by exhaustive least-squares scan.

    A dependency-free stand-in for `ruptures.Binseg`: for every admissible split
    it computes the within-segment sum of squares and returns the split that
    minimises it, plus a normalised strength score.
    """
    y = np.asarray(series, dtype="float64")
    y = y[~np.isnan(y)]
    n = len(y)
    if n < 2 * min_size:
        return None

    best = None
    total_ss = ((y - y.mean()) ** 2).sum()
    for k in range(min_size, n - min_size + 1):
        a, b = y[:k], y[k:]
        ss = ((a - a.mean()) ** 2).sum() + ((b - b.mean()) ** 2).sum()
        if best is None or ss < best[1]:
            best = (k, ss)
    k, ss = best
    return {
        "index": k,
        "label": series.index[k] if hasattr(series, "index") else k,
        "mean_before": float(y[:k].mean()),
        "mean_after": float(y[k:].mean()),
        "shift": float(y[k:].mean() - y[:k].mean()),
        "variance_explained": float(1 - ss / total_ss) if total_ss else np.nan,
    }


# --------------------------------------------------------- coverage-adjusted
def coverage_adjusted_series(det, cov, freq="M", regions=None):
    """Detections per 1000 km2 of observed sea, per period and AIS status.

    Only defined where footprint coverage exists (from 2025-12 onward in this
    dataset); earlier months return NaN rather than a misleading number.
    """

    regions = regions or ["strait_of_hormuz", "persian_gulf", "gulf_of_oman"]
    obs = cvg.daily_region_coverage(cov)
    obs = obs[obs["region"].isin(regions)]
    obs["period"] = obs["date"].dt.to_period(freq).astype(str)
    obs_agg = obs.groupby("period", observed=True)["observed_km2"].sum()

    d = det[det["region"].isin(regions)].copy()
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)
    counts = (d.groupby(["period", "ais_status"], observed=True)
              .size().unstack(fill_value=0)
              .reindex(columns=C.AIS_STATUS, fill_value=0))

    out = counts.join(obs_agg.rename("observed_km2"), how="left")
    for col in C.AIS_STATUS:
        out[f"{col}_per_1000km2"] = np.where(
            out["observed_km2"] > 0, 1000 * out[col] / out["observed_km2"], np.nan)
    out["total_per_1000km2"] = np.where(
        out["observed_km2"] > 0,
        1000 * out[list(C.AIS_STATUS)].sum(axis=1) / out["observed_km2"], np.nan)
    return out.reset_index()


def coverage_gap_report(det, cov, freq="M"):
    """Where footprint coverage is missing, so normalised stats can be caveated."""
    obs = cov.groupby(cov["date"].dt.to_period(freq).astype(str))["date"].nunique()
    d = det.copy()
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)
    det_days = d.groupby("period")["timestamp"].apply(lambda s: s.dt.date.nunique())
    out = pd.DataFrame({"days_with_detections": det_days,
                        "days_with_footprints": obs}).fillna(0).astype(int)
    out["coverage_available"] = out["days_with_footprints"] > 0
    return out.reset_index(names="period")


def window_days(det, periods):
    """Distinct calendar days with detections in a window.

    Reported alongside any pre/post comparison because the exports are not all
    full months: February 2026 stops on the 25th, May on the 28th and June on
    the 29th, so even two four-month windows are not exactly equal.
    """
    d = det[det["timestamp"].dt.to_period("M").astype(str).isin(periods)]
    return int(d["timestamp"].dt.date.nunique())


def window_summary(det, pre=None, post=None):
    """Side-by-side sanity table for the two comparison windows."""
    pre = pre or C.PRE_MATCHED
    post = post or C.POST_WINDOW
    rows = []
    for label, periods in [("pre", pre), ("post", post)]:
        d = det[det["timestamp"].dt.to_period("M").astype(str).isin(periods)]
        rows.append({
            "window": label,
            "months": len(periods),
            "first": str(d["timestamp"].min()),
            "last": str(d["timestamp"].max()),
            "days_with_detections": window_days(det, periods),
            "detections": len(d),
        })
    out = pd.DataFrame(rows)
    out["detections_per_day"] = (out["detections"]
                                 / out["days_with_detections"]).round(1)
    return out

In [9]:
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go


STATUS_LABEL = {
    "matched": "AIS matched",
    "weak_match": "Weak AIS match",
    "no_ais_candidate": "No AIS candidate (dark)",
}


# --------------------------------------------------------------------- layout
def _base_layout(title, subtitle=None, height=640, legend=True):
    """Common layout.

    The title is anchored to the figure *container* and the legend to the top of
    the plotting area, with the top margin sized to hold both.  Anchoring both
    to the same band - plotly's default - makes a horizontal legend overprint
    the title as soon as there are more than about three series.
    """
    text = f"<b>{title}</b>"
    if subtitle:
        text += f"<br><span style='font-size:13px;color:{C.INK_MUTED}'>{subtitle}</span>"
    top = 132 if subtitle else 108
    return dict(
        title=dict(text=text, x=0.012, xanchor="left", xref="container",
                   y=0.955, yanchor="top", yref="container",
                   font=dict(size=19, color=C.INK)),
        paper_bgcolor=C.SURFACE,
        plot_bgcolor=C.SURFACE,
        font=dict(color=C.INK, family="Segoe UI, Inter, system-ui, sans-serif", size=13),
        height=height,
        margin=dict(l=64, r=28, t=top, b=60),
        showlegend=legend,
        legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0,
                    font=dict(color=C.INK, size=12),
                    orientation="h", yanchor="bottom", y=1.015,
                    xanchor="right", x=1.0),
        hoverlabel=dict(bgcolor="#16202e", font=dict(color=C.INK, size=12),
                        bordercolor=C.GRID),
    )


def style_axes(fig, xtitle=None, ytitle=None, xtype=None):
    fig.update_xaxes(
        title_text=xtitle, gridcolor=C.GRID, zeroline=False,
        linecolor=C.GRID, tickfont=dict(color=C.INK_MUTED),
        title_font=dict(color=C.INK_MUTED, size=12), type=xtype,
    )
    fig.update_yaxes(
        title_text=ytitle, gridcolor=C.GRID, zeroline=False,
        linecolor=C.GRID, tickfont=dict(color=C.INK_MUTED),
        title_font=dict(color=C.INK_MUTED, size=12),
    )
    return fig


def save(fig, name, also_png=False):
    """Write an interactive HTML figure (and optionally a PNG) to outputs."""
    path = C.FIG_DIR / f"{name}.html"
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    if also_png:
        try:
            fig.write_image(C.FIG_DIR / f"{name}.png", scale=2)
        except Exception as exc:      # kaleido not installed - HTML is enough
            print(f"    (png skipped for {name}: {exc})")
    return path


# ----------------------------------------------------------------------- maps
def geo_layout(box=None, theme=None):
    box = box or C.AOI
    theme = theme or C.MAP_THEME
    lon_min, lat_min, lon_max, lat_max = box
    return dict(
        projection_type="mercator",
        showland=True, landcolor=theme["land"],
        showocean=True, oceancolor=theme["ocean"],
        showcountries=True, countrycolor=theme["country"], countrywidth=0.6,
        showcoastlines=True, coastlinecolor=theme["coastline"], coastlinewidth=0.8,
        showlakes=True, lakecolor=theme["ocean"],
        bgcolor=theme["bg"],
        lonaxis_range=[lon_min, lon_max], lataxis_range=[lat_min, lat_max],
        resolution=50,
    )


def map_detections(df, title, subtitle=None, box=None, colour_by="ais_status",
                   size=3.4, opacity=0.75, height=680, max_points=120_000,
                   order=None, colours=None, labels=None):
    """Scatter every detection on a real map, coloured by a categorical column.

    Defaults to AIS status. Pass ``colour_by="size_band"`` with the size-band
    order and colours to show the fleet by hull-length class instead.
    """
    d = df
    if len(d) > max_points:
        d = d.sample(max_points, random_state=0)
        subtitle = ((subtitle + "  |  ") if subtitle else "") + \
                   f"showing a {max_points:,}-point sample of {len(df):,}"

    if order is None:
        order = C.AIS_STATUS if colour_by == "ais_status" else \
            list(pd.Series(d[colour_by]).dropna().unique())
    colours = colours or C.CAT_COLOURS
    labels = labels or (STATUS_LABEL if colour_by == "ais_status" else {})
    present = set(d[colour_by].dropna())

    fig = go.Figure()
    for key in order:
        if key not in present:
            continue
        sub = d[d[colour_by] == key]
        fig.add_trace(go.Scattergeo(
            lon=sub["lon"], lat=sub["lat"], mode="markers",
            name=f"{labels.get(key, key)} ({len(sub):,})",
            marker=dict(size=size, opacity=opacity,
                        color=colours.get(key, "#888"),
                        line=dict(width=0)),
            hovertemplate=("%{lat:.3f}, %{lon:.3f}<br>"
                           + str(labels.get(key, key)) + "<extra></extra>"),
        ))
    fig.update_layout(**_base_layout(title, subtitle, height=height))
    fig.update_geos(**geo_layout(box))
    return fig


def vessel_group(series):
    """Map raw GFW categories onto the four display groups."""
    return (series.astype(str).map(C.VESSEL_GROUPS)
            .fillna("Specialised / other"))


def _segment_xy(s):
    """Flatten segments into one lon/lat pair-list separated by NaN gaps."""
    n = len(s)
    lons = np.empty(n * 3, dtype="float64")
    lats = np.empty(n * 3, dtype="float64")
    lons[0::3] = s["lon"].to_numpy()
    lons[1::3] = s["lon2"].to_numpy()
    lons[2::3] = np.nan
    lats[0::3] = s["lat"].to_numpy()
    lats[1::3] = s["lat2"].to_numpy()
    lats[2::3] = np.nan
    return lons, lats


def drop_antimeridian(seg, threshold=180.0):
    """Remove segments whose endpoints straddle the +/-180 meridian.

    A segment from 179E to 179W is a short hop across the date line, but drawn
    in plate-carree coordinates it becomes a line straight across the entire
    map. At world scale these false streaks are misleading. Returns
    (kept, n_dropped).
    """
    span = (seg["lon2"] - seg["lon"]).abs()
    keep = span <= threshold
    return seg[keep], int((~keep).sum())


def map_track_lines(seg, title, subtitle=None, box=None, colour=None,
                    width=1.7, opacity=0.45, height=680, max_segments=25_000,
                    colour_by=None, handle_antimeridian=True,
                    group_order=None, group_colours=None):
    """Draw reconstructed vessel segments as polylines on a map.

    The default width and opacity are deliberately heavier than a normal line
    chart. At 1 px and 0.28 opacity the corridors read as haze rather than
    routes, particularly once the figure is scaled down onto a report page.

    Segments are flattened into one trace per colour group, separated by NaN
    gaps: a single trace holding 75,000 points renders instantly, where 25,000
    separate two-point traces would not render at all.

    ``colour_by="vessel_group"`` splits the lines by vessel type. The three
    chromatic groups use categorical slots 1-3 (validated all-pairs on this
    surface); the pooled long tail gets a deliberately recessive grey.
    """
    s = seg
    if handle_antimeridian:
        s, n_dropped = drop_antimeridian(s)
        if n_dropped:
            subtitle = ((subtitle + "  |  ") if subtitle else "") + \
                       f"{n_dropped} date-line crossing(s) omitted"
    if len(s) > max_segments:
        s = s.sample(max_segments, random_state=0)
        subtitle = ((subtitle + "  |  ") if subtitle else "") + \
                   f"{max_segments:,}-segment sample of {len(seg):,}"

    fig = go.Figure()

    if colour_by == "vessel_group":
        s = s.copy()
        s["_grp"] = vessel_group(s["matched_category"])
        order, colours = C.VESSEL_GROUP_ORDER, C.VESSEL_GROUP_COLOURS
    elif colour_by is not None:
        # Any other categorical column (e.g. a size band); the caller supplies
        # the order and colours because they belong to that column's domain.
        s = s.copy()
        s["_grp"] = s[colour_by]
        order = group_order or list(pd.Series(s["_grp"]).dropna().unique())
        colours = group_colours or {}

    if colour_by is not None:
        counts = s["_grp"].value_counts()
        for grp in order:
            sub = s[s["_grp"] == grp]
            if sub.empty:
                continue
            lons, lats = _segment_xy(sub)
            fig.add_trace(go.Scattergeo(
                lon=lons, lat=lats, mode="lines",
                name=f"{grp} ({counts.get(grp, 0):,})",
                line=dict(width=width, color=colours.get(grp, "#888")),
                opacity=opacity, hoverinfo="skip",
            ))
        legend = True
    else:
        lons, lats = _segment_xy(s)
        fig.add_trace(go.Scattergeo(
            lon=lons, lat=lats, mode="lines", name="Vessel segment",
            line=dict(width=width, color=colour or C.CAT_COLOURS["matched"]),
            opacity=opacity, hoverinfo="skip", showlegend=False,
        ))
        legend = False

    fig.update_layout(**_base_layout(title, subtitle, height=height, legend=legend))
    fig.update_geos(**geo_layout(box))
    return fig


def heat_scale(ramp=None):
    """Sequential scale for a dark map: zero recedes, maximum is brightest.

    Defaults to the single-hue blue ramp read dark-to-light. Pass C.SEQ_HEAT for
    the warm alternative.
    """
    return [[v, c] for v, c in (ramp or C.SEQ_BLUE_DARK)]


def diverging_scale():
    """Signed-change scale: blue below zero, warm above, dark neutral midpoint."""
    return [[v, c] for v, c in C.DIV_BLUE_YELLOW_RED]


def map_change(grid, title, subtitle=None, box=None, value="delta",
               height=660, size=6, colourbar_title=None, qmax=0.98, limit=None):
    """Map of a signed quantity on the diverging scale.

    The colour range is clipped symmetrically at the ``qmax`` quantile of the
    absolute value, so zero stays at the neutral midpoint - an asymmetric range
    would put "no change" somewhere on the blue or the warm arm and make the
    whole map read as though it had shifted.
    """
    lim = limit or float(np.nanquantile(np.abs(grid[value]), qmax)) or 1.0
    fig = go.Figure(go.Scattergeo(
        lon=grid["lon"], lat=grid["lat"], mode="markers",
        marker=dict(size=size, color=grid[value], cmin=-lim, cmax=lim,
                    colorscale=diverging_scale(), line=dict(width=0),
                    colorbar=dict(title=colourbar_title or "Change",
                                  tickfont=dict(color=C.INK_MUTED),
                                  title_font=dict(color=C.INK_MUTED))),
        hovertemplate="%{lat:.2f}, %{lon:.2f}<br>%{marker.color:.2f}<extra></extra>",
    ))
    fig.update_layout(**_base_layout(title, subtitle, height=height, legend=False))
    fig.update_geos(**geo_layout(box))
    return fig


def map_density(grid, title, subtitle=None, box=None, value="per_obs_day",
                height=680, size=7, colourbar_title=None, qmax=0.98, vmax=None):
    """Grid-cell density map - the aggregate counterpart to the scatter maps.

    A scatter of every detection shows individual ships but saturates wherever
    traffic is heavy, which is where the relevant variation lies. This
    bins to the coverage grid and colours by rate, so dense water stays readable.

    The colour scale is clipped at the ``qmax`` quantile: a handful of extreme
    anchorage cells would otherwise compress every other cell into one shade.
    """
    vmax = vmax or float(np.nanquantile(grid[value], qmax)) or 1.0

    fig = go.Figure(go.Scattergeo(
        lon=grid["lon"], lat=grid["lat"], mode="markers",
        marker=dict(size=size, color=grid[value], cmin=0, cmax=vmax,
                    colorscale=heat_scale(), line=dict(width=0),
                    colorbar=dict(title=colourbar_title or "Detections<br>per observed day",
                                  tickfont=dict(color=C.INK_MUTED),
                                  title_font=dict(color=C.INK_MUTED))),
        hovertemplate="%{lat:.2f}, %{lon:.2f}<br>%{marker.color:.2f}<extra></extra>",
    ))
    fig.update_layout(**_base_layout(title, subtitle, height=height, legend=False))
    fig.update_geos(**geo_layout(box))
    return fig


def box_aspect(box):
    """Width/height of a lon-lat box once projected with Mercator."""
    lon_min, lat_min, lon_max, lat_max = box
    y = lambda lat: np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))
    return np.radians(lon_max - lon_min) / (y(lat_max) - y(lat_min))


_COUNT_SUFFIX = re.compile(r"\s*\([\d,]+\)$")


def map_panels(panels, title, subtitle=None, box=None, ncols=2, width=1300,
               share_colourbar=True, legend=True, hspace=0.03, vspace=0.09,
               title_band=150):
    """Lay several single-map figures out as one gridded figure.

    ``panels`` is a list of ``(panel_title, fig)`` where each ``fig`` came from
    one of the map builders above. Traces are copied onto their own geo axis so
    the panels share projection, extent and styling exactly, so that the
    panels are directly comparable.

    Panel height is derived from the box's projected aspect so the map fills
    its cell instead of floating in dead space. Legend entries are merged by
    name with the per-panel counts stripped, so a four-panel ladder shows one
    legend. With ``share_colourbar`` a density grid keeps a single colour bar,
    which is only honest when every panel was built with the same limits.
    """
    from plotly.subplots import make_subplots

    box = box or C.AOI
    n = len(panels)
    nrows = -(-n // ncols)
    margin = dict(l=20, r=70 if share_colourbar else 20, t=title_band, b=20)
    panel_w = (width - margin["l"] - margin["r"]) * (1 - hspace * (ncols - 1)) / ncols
    panel_h = panel_w / box_aspect(box)
    grid_h = panel_h * nrows / (1 - vspace * (nrows - 1)) if nrows > 1 else panel_h
    height = grid_h + margin["t"] + margin["b"] + 30 * nrows   # room for panel titles

    fig = make_subplots(rows=nrows, cols=ncols,
                        specs=[[{"type": "geo"}] * ncols for _ in range(nrows)],
                        subplot_titles=[p[0] for p in panels],
                        horizontal_spacing=hspace, vertical_spacing=vspace)

    seen = set()
    for k, (_, src) in enumerate(panels):
        r, c = divmod(k, ncols)
        for tr in src.data:
            tr = tr.to_plotly_json()
            tr.pop("geo", None)
            name = tr.get("name")
            if tr.get("showlegend", True) and name is not None:
                base = _COUNT_SUFFIX.sub("", name)
                tr["name"] = base
                tr["legendgroup"] = base
                tr["showlegend"] = base not in seen
                seen.add(base)
            marker = tr.get("marker") or {}
            if "colorscale" in marker or "colorbar" in marker:
                keep = (k == 0) if share_colourbar else True
                marker["showscale"] = keep
                if keep:
                    cb = marker.setdefault("colorbar", {})
                    cb.update(dict(len=0.55, thickness=12, x=1.005))
                tr["marker"] = marker
            fig.add_trace(go.Scattergeo(**tr), row=r + 1, col=c + 1)

    fig.update_layout(**_base_layout(title, subtitle, height=height, legend=legend))
    fig.update_layout(width=width, margin=margin,
                      legend=dict(y=1.02, yanchor="bottom", x=1.0))
    fig.update_geos(**geo_layout(box))
    for ann in fig.layout.annotations:
        ann.font.color = C.INK
        ann.font.size = 13
    return fig


def length_histogram(frames, title, subtitle=None, bands=None, band_colours=None,
                     bin_width=10, max_length=450, height=520, colours=None):
    """Distribution of radar-derived hull length with the size classes shaded.

    ``frames`` maps a series name to a detections frame. The class bands are
    drawn as background strips so the reader can see how much of each fleet
    sits in each class before the class-by-class maps that follow.
    """
    bands = bands or []
    fig = go.Figure()
    for lo, hi, label in bands:
        hi_draw = min(hi, max_length)
        fig.add_vrect(x0=lo, x1=hi_draw,
                      fillcolor=(band_colours or {}).get(label, "#888"),
                      opacity=0.10, line_width=0, layer="below")
        fig.add_annotation(x=(lo + hi_draw) / 2, y=1.0, yref="paper", yanchor="top",
                           text=label.split(" (")[0], showarrow=False,
                           font=dict(size=11, color=C.INK_MUTED))
    colours = colours or {}
    edges = np.arange(0, max_length + bin_width, bin_width)
    for name, df in frames.items():
        length = df["length_m"].dropna().clip(upper=max_length - 0.01)
        counts, _ = np.histogram(length, bins=edges)
        share = 100 * counts / max(counts.sum(), 1)
        fig.add_trace(go.Scatter(
            x=edges[:-1] + bin_width / 2, y=share, mode="lines", name=name,
            line=dict(width=2.2, color=colours.get(name), shape="hvh"),
            hovertemplate=f"{name}: %{{y:.1f}}% at %{{x}} m<extra></extra>",
        ))
    fig.update_layout(**_base_layout(title, subtitle, height=height))
    style_axes(fig, xtitle="Radar-derived length (m)", ytitle="Share of detections (%)")
    fig.update_xaxes(range=[0, max_length])
    return fig


def add_ports(fig, box=None, colour="#e8e2d4", size=7, labels=True):
    """Overlay the port reference layer.

    ``labels=False`` keeps the markers but drops the text, for dense maps where
    two dozen port names collide into an unreadable mat.  The names stay
    available on hover either way.
    """
    box = box or C.AOI
    lon_min, lat_min, lon_max, lat_max = box
    names, lons, lats = [], [], []
    for name, (lon, lat) in C.PORTS.items():
        if lon_min <= lon <= lon_max and lat_min <= lat <= lat_max:
            names.append(name)
            lons.append(lon)
            lats.append(lat)
    if not names:
        return fig
    fig.add_trace(go.Scattergeo(
        lon=lons, lat=lats, mode="markers+text" if labels else "markers",
        name="Port", text=names, textposition="top center",
        textfont=dict(size=9, color=C.INK_MUTED),
        marker=dict(size=size, color=colour, symbol="square",
                    line=dict(width=1, color=C.SURFACE)),
        hovertemplate="%{text}<extra></extra>", showlegend=False,
    ))
    return fig


def add_box(fig, box, name, colour="#e8e2d4", width=1.4, dash="dot"):
    """Outline a study region on a map."""
    lon_min, lat_min, lon_max, lat_max = box
    fig.add_trace(go.Scattergeo(
        lon=[lon_min, lon_max, lon_max, lon_min, lon_min],
        lat=[lat_min, lat_min, lat_max, lat_max, lat_min],
        mode="lines", name=name, showlegend=False,
        line=dict(color=colour, width=width, dash=dash),
        hoverinfo="skip",
    ))
    return fig


# --------------------------------------------------------------- time series
def line_series(frame, x, y, colour_col, title, subtitle=None, ytitle=None,
                xtitle=None, colours=None, labels=None, height=520,
                direct_label=True):
    """Multi-series line chart with a crosshair hover and direct end-labels."""
    colours = colours or C.CAT_COLOURS
    labels = labels or STATUS_LABEL
    fig = go.Figure()
    series = list(dict.fromkeys(frame[colour_col]))

    for key in series:
        sub = frame[frame[colour_col] == key].sort_values(x)
        fig.add_trace(go.Scatter(
            x=sub[x], y=sub[y], mode="lines+markers",
            name=labels.get(key, str(key)),
            line=dict(width=2, color=colours.get(key)),
            marker=dict(size=5, line=dict(width=0)),
            hovertemplate=f"{labels.get(key, key)}: %{{y:,.4g}}<extra></extra>",
        ))

    # Direct labels are legible only for a handful of series.
    if direct_label and len(series) <= 4:
        for key in series:
            sub = frame[frame[colour_col] == key].sort_values(x)
            if sub.empty:
                continue
            fig.add_annotation(
                x=sub[x].iloc[-1], y=sub[y].iloc[-1],
                text=f" {labels.get(key, str(key))}", showarrow=False,
                xanchor="left", font=dict(color=colours.get(key, C.INK), size=11),
            )

    fig.update_layout(**_base_layout(title, subtitle, height=height))
    fig.update_layout(hovermode="x unified")
    if direct_label and len(series) <= 4:
        # Reserve room for the end-labels so they do not clip off the canvas.
        width = max(len(labels.get(k, str(k))) for k in series)
        fig.update_layout(margin_r=max(28, int(width * 7.2) + 24))
    style_axes(fig, xtitle, ytitle)
    return fig


def add_event_bands(fig, events):
    """Shade named date windows behind a time series.

    Uses explicit shapes rather than ``add_vrect``: these charts have a
    categorical x axis (month strings), and plotly's vrect/vline helpers try to
    average the axis range numerically to place their annotation, which raises
    on category values.
    """
    for label, (start, end) in events.items():
        fig.add_shape(type="rect", x0=start, x1=end, y0=0, y1=1,
                      xref="x", yref="paper", layer="below",
                      fillcolor="#e8e2d4", opacity=0.07, line_width=0)
        fig.add_annotation(x=start, y=1.0, xref="x", yref="paper",
                           text=label, showarrow=False, xanchor="left",
                           yanchor="bottom",
                           font=dict(size=10, color=C.INK_MUTED))
    return fig


def add_pipeline_break(fig, when=None, label="pipeline v3 to v4"):
    """Mark the GFW pipeline v3 -> v4 boundary, which confounds raw counts."""
    when = when or C.PIPELINE_BREAK
    fig.add_shape(type="line", x0=when, x1=when, y0=0, y1=1,
                  xref="x", yref="paper", layer="above",
                  line=dict(color=C.INK_MUTED, width=1.4, dash="dash"))
    # Sits just inside the plot area: the band above it belongs to the legend.
    fig.add_annotation(x=when, y=0.985, xref="x", yref="paper",
                       text=f" {label}", showarrow=False,
                       xanchor="left", yanchor="top",
                       font=dict(size=10, color=C.INK_MUTED))
    return fig


def paired_panels(left, right, x, y, colour_col, title, subtitle=None,
                  left_title="", right_title="", ytitle_left=None,
                  ytitle_right=None, xtitle=None, colours=None, labels=None,
                  height=560):
    """Two line panels sharing one legend, colour scheme and x axis.

    Used where the same series is shown on two different scales - raw counts
    beside a normalised rate, for instance. Plotting those as two separate
    figures suggests a difference that is not present, and putting them on one
    pair of axes would require dual axes. Side-by-side panels with a shared
    legend show that the shape is unchanged.
    """
    from plotly.subplots import make_subplots

    colours = colours or C.CAT_COLOURS
    labels = labels or STATUS_LABEL

    fig = make_subplots(rows=1, cols=2, shared_xaxes=False,
                        horizontal_spacing=0.09,
                        subplot_titles=(left_title, right_title))

    for col, (frame, ytitle) in enumerate(
            [(left, ytitle_left), (right, ytitle_right)], start=1):
        for key in dict.fromkeys(frame[colour_col]):
            sub = frame[frame[colour_col] == key].sort_values(x)
            fig.add_trace(go.Scatter(
                x=sub[x], y=sub[y], mode="lines+markers",
                name=labels.get(key, str(key)),
                legendgroup=str(key), showlegend=(col == 1),
                line=dict(width=2, color=colours.get(key)),
                marker=dict(size=5, line=dict(width=0)),
                hovertemplate=f"{labels.get(key, key)}: %{{y:,.4g}}<extra></extra>",
            ), row=1, col=col)
        fig.update_yaxes(title_text=ytitle, row=1, col=col)
        fig.update_xaxes(title_text=xtitle, row=1, col=col)

    fig.update_layout(**_base_layout(title, subtitle, height=height))
    # make_subplots puts its panel titles at the top of each subplot domain,
    # which is exactly where the shared legend sits. Lift the legend clear and
    # widen the top margin to hold title, legend and panel titles in three bands.
    fig.update_layout(hovermode="x unified", margin_t=170,
                      legend=dict(y=1.10, yanchor="bottom"))
    fig.update_xaxes(gridcolor=C.GRID, zeroline=False, linecolor=C.GRID,
                     tickfont=dict(color=C.INK_MUTED),
                     title_font=dict(color=C.INK_MUTED, size=12))
    fig.update_yaxes(gridcolor=C.GRID, zeroline=False, linecolor=C.GRID,
                     tickfont=dict(color=C.INK_MUTED),
                     title_font=dict(color=C.INK_MUTED, size=12))
    for ann in fig.layout.annotations:
        ann.font.color = C.INK
        ann.font.size = 12
    return fig

In [10]:
import numpy as np
import pandas as pd


EARTH_R_KM = 6371.0088
KN_PER_KMH = 0.539957


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between arrays of points, in km."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * EARTH_R_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def build_segments(df, max_gap_h=96, max_speed_kn=35, min_obs=2):
    """Join consecutive same-MMSI detections into plausible track segments.

    A segment is kept only if the implied speed between its endpoints is under
    ``max_speed_kn``; faster implies the two detections are different vessels
    sharing a spoofed or mis-assigned MMSI, or a matching error.
    """
    d = df[(df["ais_status"] == "matched") & df["mmsi"].notna()].copy()
    d = d.sort_values(["mmsi", "timestamp"])
    if d.empty:
        return pd.DataFrame()

    grp = d.groupby("mmsi", observed=True)
    d["lat2"] = grp["lat"].shift(-1)
    d["lon2"] = grp["lon"].shift(-1)
    d["t2"] = grp["timestamp"].shift(-1)

    seg = d.dropna(subset=["lat2", "lon2", "t2"]).copy()
    seg["gap_h"] = (seg["t2"] - seg["timestamp"]).dt.total_seconds() / 3600.0
    seg["dist_km"] = haversine_km(seg["lat"], seg["lon"], seg["lat2"], seg["lon2"])
    seg["speed_kn"] = np.where(
        seg["gap_h"] > 0, seg["dist_km"] / seg["gap_h"] * KN_PER_KMH, np.nan)

    seg = seg[(seg["gap_h"] > 0) & (seg["gap_h"] <= max_gap_h)
              & (seg["speed_kn"] <= max_speed_kn)]

    if min_obs > 2:
        counts = seg.groupby("mmsi", observed=True)["dist_km"].transform("size")
        seg = seg[counts >= (min_obs - 1)]

    keep = ["mmsi", "timestamp", "t2", "lat", "lon", "lat2", "lon2",
            "gap_h", "dist_km", "speed_kn", "length_m", "matched_category", "region"]
    return seg[[c for c in keep if c in seg.columns]].reset_index(drop=True)


def bearing_deg(lat1, lon1, lat2, lon2):
    """Initial great-circle bearing, degrees clockwise from north."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


def gate_transits(df, gate=None, freq="M"):
    """Distinct vessels observed inside the strait gate, per period.

    Counts unique MMSIs (identity-based, so a vessel imaged twice in a month is
    one transit-capable presence) alongside raw detections and dark detections,
    which have no identity and can only be counted as observations.
    """
    gate = gate or C.HORMUZ_GATE
    d = df[build.in_box(df, gate)].copy()
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)

    out = d.groupby("period", observed=True).agg(
        detections=("lat", "size"),
        unique_mmsi=("mmsi", lambda s: s.dropna().nunique()),
    )
    dark = (d[d["ais_status"] == "no_ais_candidate"]
            .groupby("period", observed=True).size().rename("dark_detections"))
    out = out.join(dark).fillna({"dark_detections": 0})
    out["dark_share"] = 100 * out["dark_detections"] / out["detections"]
    return out.reset_index()


def transit_direction(seg, gate=None):
    """Split gate-crossing segments into inbound / outbound by bearing.

    The strait runs roughly NW-SE, so a segment heading broadly north-west is
    entering the Gulf and one heading broadly south-east is leaving it.
    """
    gate = gate or C.HORMUZ_GATE
    lon_min, lat_min, lon_max, lat_max = gate
    mid_lon = (lon_min + lon_max) / 2
    mid_lat = (lat_min + lat_max) / 2

    s = seg.copy()
    near = (s["lon"].between(lon_min - 1, lon_max + 1)
            & s["lat"].between(lat_min - 1, lat_max + 1))
    s = s[near].copy()
    if s.empty:
        return s
    s["bearing"] = bearing_deg(s["lat"], s["lon"], s["lat2"], s["lon2"])
    s["direction"] = np.where(
        (s["bearing"] > 225) | (s["bearing"] < 45), "inbound (into Gulf)",
        np.where(s["bearing"].between(90, 225), "outbound (to Gulf of Oman)", "cross"))
    s["period"] = s["timestamp"].dt.to_period("M").astype(str)
    return s


# ---------------------------------------------------------------- loitering
def find_loitering(df, radius_km=12.0, min_span_h=36.0, min_obs=3,
                   max_speed_kn=1.5, max_gap_h=120.0):
    """Vessels repeatedly imaged in the same small area over an extended span.

    Operationalised as: >= ``min_obs`` detections of one MMSI whose successive
    positions never move further than ``radius_km``, spanning at least
    ``min_span_h`` hours, with a mean implied speed under ``max_speed_kn``.
    This is the SAR-visible signature of anchoring or waiting offshore.

    ``max_gap_h`` bounds the interval between the two detections making up a
    step.  Without it a vessel imaged in September and again in November 10 km
    away scores as "stationary" for two months, when in reality it could have
    sailed anywhere in between and returned.
    """
    d = df[(df["ais_status"] == "matched") & df["mmsi"].notna()].copy()
    d = d.sort_values(["mmsi", "timestamp"])
    if d.empty:
        return pd.DataFrame()

    grp = d.groupby("mmsi", observed=True)
    d["prev_lat"] = grp["lat"].shift()
    d["prev_lon"] = grp["lon"].shift()
    d["prev_t"] = grp["timestamp"].shift()

    step = d.dropna(subset=["prev_lat", "prev_t"]).copy()
    step["dist_km"] = haversine_km(step["prev_lat"], step["prev_lon"],
                                   step["lat"], step["lon"])
    step["gap_h"] = (step["timestamp"] - step["prev_t"]).dt.total_seconds() / 3600
    step["speed_kn"] = np.where(step["gap_h"] > 0,
                                step["dist_km"] / step["gap_h"] * KN_PER_KMH, np.nan)

    # A "stationary step" is a short hop between two closely-spaced-in-time looks.
    step["stationary"] = (
        (step["dist_km"] <= radius_km)
        & (step["speed_kn"] <= max_speed_kn)
        & (step["gap_h"] <= max_gap_h)
    )

    # Consecutive stationary steps form one loitering episode.
    step["run"] = (~step["stationary"]).groupby(step["mmsi"], observed=True).cumsum()
    epi = step[step["stationary"]].groupby(["mmsi", "run"], observed=True).agg(
        n_obs=("dist_km", "size"),
        start=("prev_t", "min"),
        end=("timestamp", "max"),
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        max_step_km=("dist_km", "max"),
        mean_speed_kn=("speed_kn", "mean"),
        length_m=("length_m", "first"),
        category=("matched_category", "first"),
    ).reset_index()

    epi["span_h"] = (epi["end"] - epi["start"]).dt.total_seconds() / 3600
    epi = epi[(epi["n_obs"] >= (min_obs - 1)) & (epi["span_h"] >= min_span_h)]
    epi["period"] = epi["start"].dt.to_period("M").astype(str)
    return epi.reset_index(drop=True)


def loitering_load(epi, freq="M"):
    """Loitering vessel-days per period, apportioned across months.

    An episode is attributed to every period it overlaps, in proportion to the
    time spent there, rather than wholly to the month it began.  Attributing by
    start month alone produces a spurious downward trend: long anchorages that
    begin early are counted at the start of the series, while episodes still
    running at the end of the data are truncated.
    """
    if epi.empty:
        return pd.DataFrame(columns=["period", "vessel_days", "episodes", "vessels"])

    rows = []
    for r in epi.itertuples(index=False):
        edges = pd.date_range(r.start.normalize(),
                              r.end.normalize() + pd.Timedelta(days=1), freq="D")
        for day in edges[:-1]:
            lo = max(r.start, day)
            hi = min(r.end, day + pd.Timedelta(days=1))
            hours = (hi - lo).total_seconds() / 3600
            if hours > 0:
                rows.append((day.to_period(freq).astype(str)
                             if hasattr(day.to_period(freq), "astype")
                             else str(day.to_period(freq)),
                             r.mmsi, hours / 24.0))

    spread = pd.DataFrame(rows, columns=["period", "mmsi", "vessel_days"])
    out = spread.groupby("period", observed=True).agg(
        vessel_days=("vessel_days", "sum"),
        vessels=("mmsi", "nunique"),
    ).reset_index()
    starts = epi.groupby("period", observed=True).size().rename("episodes_started")
    return out.merge(starts.reset_index(), on="period", how="left").fillna(
        {"episodes_started": 0})


def congestion_grid(df, cov, res=0.1, freq="M", min_obs_days=2):
    """Detections per observed cell-day, by grid cell and period.

    Dividing by the number of days the cell was actually imaged turns a raw
    detection pile-up into a comparable congestion rate, so an anchorage that
    was imaged more often does not appear as a queue.
    """

    d = df.copy()
    ix, iy = cvg.cell_index(d["lon"].to_numpy(), d["lat"].to_numpy(), res=res)
    d["ix"], d["iy"] = ix, iy
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)
    det = d.groupby(["period", "ix", "iy"], observed=True).size().rename("detections")

    c = cov.copy()
    c["period"] = c["date"].dt.to_period(freq).astype(str)
    obs = (c.groupby(["period", "ix", "iy"], observed=True)["date"]
           .nunique().rename("obs_days"))

    out = pd.concat([det, obs], axis=1).dropna(subset=["obs_days"])
    out = out[out["obs_days"] >= min_obs_days].fillna({"detections": 0})
    out["per_obs_day"] = out["detections"] / out["obs_days"]
    out = out.reset_index()
    lon, lat = cvg.cell_centres(out["ix"].to_numpy(), out["iy"].to_numpy(), res=res)
    out["lon"], out["lat"] = lon, lat
    return out


import numpy as np
import pandas as pd


# Detection-artefact labels rather than real vessel classes; excluded from
# fleet composition, kept out of the fate table by the support threshold.
ARTEFACT_CATEGORIES = {"noisy_vessel", "discrepancy"}


def type_composition(df, freq="M", min_total=400, exclude_artefacts=True):
    """Monthly counts and shares of matched detections by vessel class."""
    m = df[(df["ais_status"] == "matched") & df["matched_category"].notna()].copy()
    if exclude_artefacts:
        m = m[~m["matched_category"].astype(str).isin(ARTEFACT_CATEGORIES)]
    m["period"] = m["timestamp"].dt.to_period(freq).astype(str)

    counts = pd.crosstab(m["period"], m["matched_category"].astype(str))
    counts = counts.loc[:, counts.sum() >= min_total]
    shares = 100 * counts.div(counts.sum(axis=1), axis=0)
    return counts, shares


def type_profile(df):
    """Length and behaviour profile per class - a check that labels mean something."""
    m = df[(df["ais_status"] == "matched") & df["matched_category"].notna()]
    prof = m.groupby(m["matched_category"].astype(str), observed=True).agg(
        detections=("lat", "size"),
        vessels=("mmsi", lambda s: s.dropna().nunique()),
        median_length_m=("length_m", "median"),
        p10_length_m=("length_m", lambda s: s.quantile(0.10)),
        p90_length_m=("length_m", lambda s: s.quantile(0.90)),
        median_fishing_score=("fishing_score", "median"),
    )
    return prof.sort_values("detections", ascending=False).round(2)


def vessel_types(df, pre_periods=None):
    """Modal class per MMSI over the pre window.

    A vessel occasionally receives different labels in different scenes; taking
    the mode fixes one class per hull so the fate table is not double-counting.
    """
    pre_periods = pre_periods or C.PRE_MATCHED
    d = df[(df["ais_status"] == "matched") & df["mmsi"].notna()]
    d = d[d["timestamp"].dt.to_period("M").astype(str).isin(pre_periods)]
    d = d[d["matched_category"].notna()]
    if d.empty:
        return pd.Series(dtype=object)
    return (d.groupby("mmsi", observed=True)["matched_category"]
            .agg(lambda s: s.mode().iat[0] if len(s.mode()) else None))


def fleet_fate(global_df, box, pre_periods=None, post_periods=None, min_support=60):
    """Where the vessels of one region went, by vessel class.

    ``global_df`` must be worldwide: the test is whether a vessel absent from
    the region is present anywhere else.
    """
    pre_periods = pre_periods or C.PRE_MATCHED
    post_periods = post_periods or C.POST_WINDOW

    world_period = global_df["timestamp"].dt.to_period("M").astype(str)
    region = global_df[build.in_box(global_df, box)]
    region_period = region["timestamp"].dt.to_period("M").astype(str)

    types = vessel_types(region, pre_periods)
    if types.empty:
        return pd.DataFrame()

    seen_here = set(region.loc[region_period.isin(post_periods)
                               & region["mmsi"].notna(), "mmsi"])
    seen_world = set(global_df.loc[world_period.isin(post_periods)
                                   & global_df["mmsi"].notna(), "mmsi"])

    fate = pd.DataFrame({"vessel_type": types.astype(str)})
    idx = fate.index
    fate["stayed"] = idx.isin(seen_here)
    fate["elsewhere"] = (~fate["stayed"]) & idx.isin(seen_world)
    fate["vanished"] = (~fate["stayed"]) & (~idx.isin(seen_world))

    out = fate.groupby("vessel_type", observed=True).agg(
        vessels=("stayed", "size"),
        stayed=("stayed", "sum"),
        elsewhere=("elsewhere", "sum"),
        vanished=("vanished", "sum"),
    )
    out = out[out["vessels"] >= min_support]
    for col in ("stayed", "elsewhere", "vanished"):
        out[f"{col}_pct"] = (100 * out[col] / out["vessels"]).round(1)
    return out.sort_values("vessels", ascending=False)


def fleet_fate_table(global_df, treatment_box=None, controls=None, **kw):
    """Fate by class for the Gulf and each control region, stacked for comparison.

    The control rows are what make the treatment rows readable: fishing vessels
    show a high 'vanished' rate everywhere, because they are small, local and
    poorly detected, so only the gap against a control carries information.
    """
    treatment_box = treatment_box or (47.0, 22.0, 62.0, 31.0)
    controls = controls or {"SE Asia": (100.0, -10.0, 125.0, 20.0),
                            "W Mediterranean": (-5.0, 35.0, 16.0, 45.0)}
    frames = []
    for name, box in [("Gulf (treatment)", treatment_box), *controls.items()]:
        f = fleet_fate(global_df, box, **kw)
        if f.empty:
            continue
        f = f.reset_index()
        f.insert(0, "region", name)
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


def relocation_contrast(fate_table, vessel_type="cargo"):
    """Treatment-minus-control gap in each outcome, for one vessel class."""
    sub = fate_table[fate_table["vessel_type"] == vessel_type]
    if sub.empty:
        return pd.DataFrame()
    treat = sub[sub["region"] == "Gulf (treatment)"]
    ctrl = sub[sub["region"] != "Gulf (treatment)"]
    if treat.empty or ctrl.empty:
        return pd.DataFrame()
    rows = []
    for col in ("stayed_pct", "elsewhere_pct", "vanished_pct"):
        rows.append({
            "outcome": col.replace("_pct", ""),
            "gulf": float(treat[col].iloc[0]),
            "control_mean": round(float(ctrl[col].mean()), 1),
            "gap_pp": round(float(treat[col].iloc[0] - ctrl[col].mean()), 1),
        })
    return pd.DataFrame(rows)


# ------------------------------------------------------- size-based filtering
# Vessels that move seaborne trade are large. A modern containership runs from
# roughly 140 m (early cellular) through 250 m (Panamax) to 400 m (ULCS), and
# the tankers that dominate Gulf traffic are comparable. A 25 m radar return is
# a dhow, a fishing boat or a workboat, which is not what a trade-disruption
# study is about.
#
# Length matters here for a second, less obvious reason. `matched_category` is
# only meaningful for detections that matched AIS - every dark detection carries
# the literal value 'unmatched' - so filtering by vessel class silently deletes
# the entire dark population and makes any dark-share comparison impossible.
# `length_m` is radar-derived and present for *every* detection, matched or not,
# so it is the only attribute that can filter both sides of that comparison.
LENGTH_BANDS = {
    "all": 0,
    "over 50 m": 50,
    "commercial (100 m+)": 100,
    "large commercial (150 m+)": 150,
}
COMMERCIAL_MIN_LENGTH_M = 100


def commercial(df, min_length_m=COMMERCIAL_MIN_LENGTH_M):
    """Detections of vessels at or above a length threshold."""
    return df[df["length_m"] >= min_length_m]


def size_profile_by_status(df, label=""):
    """Median length and large-vessel share per AIS status.

    Computed on the study region rather than the whole AOI: the two differ
    materially, and only the regional profile bears on Gulf shipping.
    """
    out = df.groupby("ais_status", observed=True)["length_m"].agg(
        detections="size", median_length_m="median",
        p25=lambda s: s.quantile(0.25), p75=lambda s: s.quantile(0.75))
    out["pct_over_100m"] = (100 * df.assign(big=df["length_m"] >= 100)
                            .groupby("ais_status", observed=True)["big"].mean())
    out.insert(0, "scope", label or "region")
    return out.round(1)


def dark_share_by_threshold(df, thresholds=None, freq="M"):
    """Monthly dark share recomputed at each length threshold.

    A robustness check on the headline result: if the rise in dark detections
    were an artefact of small craft, restricting to large vessels would flatten
    it.
    """
    thresholds = thresholds or LENGTH_BANDS
    d = df.copy()
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)
    cols = {}
    for label, thr in thresholds.items():
        sub = d[d["length_m"] >= thr]
        cols[label] = (sub.groupby("period", observed=True)["ais_status"]
                       .apply(lambda s: 100 * (s == "no_ais_candidate").mean()))
    return pd.DataFrame(cols).round(1)


def retention_by_threshold(df, thresholds=None):
    """How much of the population each threshold keeps, and its dark share."""
    thresholds = thresholds or LENGTH_BANDS
    rows = []
    for label, thr in thresholds.items():
        sub = df[df["length_m"] >= thr]
        rows.append({
            "filter": label, "min_length_m": thr, "detections": len(sub),
            "pct_of_all": round(100 * len(sub) / max(len(df), 1), 1),
            "dark_share_pct": round(100 * (sub["ais_status"] == "no_ais_candidate").mean(), 1),
            "median_length_m": round(float(sub["length_m"].median()), 1),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------- containership size classes
# Band edges follow the standard containership generations rather than round
# numbers, so each band corresponds to a real class of ship (Rodrigue, The
# Geography of Transport Systems, "Evolution of Containerships Classes"):
#
#   early cellular  ~137-215 m      Panamax        ~250-290 m
#   Panamax max     ~290 m          post-Panamax   ~300-340 m
#   New-Panamax     ~366 m          VLCS / ULCS    ~397-400 m
#
# Radar-derived length varies with sea state and aspect, so a hull near a
# boundary can fall either side of it. The bands are wide enough - 100 m each -
# that this blurs individual assignments without moving the aggregate pattern.
SIZE_BANDS = [
    (0,   100,        "Under 100 m"),
    (100, 200,        "100-200 m (feeder / handysize)"),
    (200, 300,        "200-300 m (Panamax class)"),
    (300, float("inf"), "300 m+ (post-Panamax and larger)"),
]
SIZE_BAND_ORDER = [b[2] for b in SIZE_BANDS]
# Ordinal ramp, widened for separability: four adjacent steps of the documented
# blue scale are too close to trace apart as lines, so these span most of it.
# Every step clears 3:1 against the dark surface.
SIZE_BAND_COLOURS = dict(zip(SIZE_BAND_ORDER,
                             ["#256abf", "#3987e5", "#86b6ef", "#cde2fb"]))


def assign_size_band(df, bands=None):
    """Label each detection with its containership-class size band."""
    bands = bands or SIZE_BANDS
    lo = [b[0] for b in bands] + [float("inf")]
    labels = [b[2] for b in bands]
    return pd.cut(df["length_m"], bins=lo, labels=labels,
                  right=False, include_lowest=True)


def size_band_dark_share(df, freq="M", bands=None):
    """Monthly dark share within each size band."""
    d = df.copy()
    d["size_band"] = assign_size_band(d, bands)
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)
    out = (d.dropna(subset=["size_band"])
           .groupby(["period", "size_band"], observed=True)["ais_status"]
           .apply(lambda s: 100 * (s == "no_ais_candidate").mean())
           .unstack())
    return out.reindex(columns=[b for b in SIZE_BAND_ORDER if b in out.columns]).round(1)


def size_gradient(df, pre_periods=None, post_periods=None, bands=None):
    """Pre/post dark share and swing per size band.

    The informative feature is the ordering: if the change were an artefact of
    small-craft noise the swing would shrink with size.
    """
    pre_periods = pre_periods or C.PRE_MATCHED
    post_periods = post_periods or C.POST_WINDOW
    d = df.copy()
    d["size_band"] = assign_size_band(d, bands)
    d["period"] = d["timestamp"].dt.to_period("M").astype(str)

    rows = []
    for band in SIZE_BAND_ORDER:
        sub = d[d["size_band"] == band]
        if sub.empty:
            continue
        pre = sub[sub["period"].isin(pre_periods)]
        post = sub[sub["period"].isin(post_periods)]
        dark = lambda x: 100 * (x["ais_status"] == "no_ais_candidate").mean()
        rows.append({
            "size_band": band,
            "detections": len(sub),
            "pct_of_all": round(100 * len(sub) / len(d), 1),
            "dark_pre_pct": round(dark(pre), 1) if len(pre) else np.nan,
            "dark_post_pct": round(dark(post), 1) if len(post) else np.nan,
        })
    out = pd.DataFrame(rows)
    out["swing_pp"] = (out["dark_post_pct"] - out["dark_pre_pct"]).round(1)
    return out

In [11]:
for _d in (det, glob_det):
    _d["size_band"] = assign_size_band(_d)
    _d["period"] = period_of(_d)
gulf = det[det["region"].isin(GULF_REGIONS)]
SHORT_BAND = {b: b.split(" (")[0] for b in SIZE_BAND_ORDER}

comp = []
for name, df in [("World", glob_det), ("Gulf", gulf)]:
    s = df[df.period.isin(PRE_MATCHED + POST_WINDOW)]
    comp.append((100 * s.size_band.value_counts(normalize=True))
                .reindex(SIZE_BAND_ORDER).rename(name))
display(pd.concat(comp, axis=1).round(1).rename_axis("share of detections (%)"))

fig = length_histogram(
    {"World": glob_det[glob_det.period.isin(PRE_MATCHED + POST_WINDOW)],
     "Gulf of Oman - Hormuz - Persian Gulf": gulf[gulf.period.isin(PRE_MATCHED + POST_WINDOW)]},
    "Distribution of radar-derived hull length, world and Gulf",
    "Share of detections per 10 m bin, Nov 2025 to Jun 2026. Shaded bands are the hull-length classes.",
    bands=SIZE_BANDS, band_colours=SIZE_BAND_COLOURS,
    colours={"World": CAT_COLOURS["matched"],
             "Gulf of Oman - Hormuz - Persian Gulf": CAT_COLOURS["no_ais_candidate"]})
save(fig, "fig02_length_histogram", also_png=True)
fig.show()

,World,Gulf
share of detections (%),,
Under 100 m,63.4,55.0
100-200 m (feeder / handysize),25.7,29.5
200-300 m (Panamax class),9.1,12.4
300 m+ (post-Panamax and larger),1.8,3.1


Two thirds of detections worldwide are under 100 m. The Gulf carries a heavier tail and a mode at
180 to 190 m, the length of a medium-range product tanker.

## 4.2 Route geography by class

Route reconstruction joins consecutive detections of one MMSI at a physically plausible speed
(under 35 knots, gaps under 96 hours). Only AIS-matched detections carry an identity, so routes
describe identifiable traffic. Each vessel is assigned to a class by the median of its radar
lengths, which is more stable than any single return.

In [12]:
def add_vessel_band(seg, source):
    """Class each segment by its vessel's median radar length."""
    med = source.groupby("mmsi", observed=True)["length_m"].median()
    seg = seg.copy()
    seg["vessel_len"] = seg["mmsi"].map(med).astype("float32")
    seg["size_band"] = assign_size_band(seg.assign(length_m=seg["vessel_len"]))
    seg["period"] = period_of(seg)
    return seg


t0 = time.time()
wseg = add_vessel_band(build_segments(glob_det), glob_det)
wseg = wseg[in_box(wseg, WORLD_BOX)]
print(f"{len(wseg):,} world segments across {wseg.mmsi.nunique():,} vessels "
      f"({time.time()-t0:.0f}s)")

sub = wseg[wseg.period.isin(PRE_MATCHED)]
panels = []
for band in SIZE_BAND_ORDER:
    s = sub[sub.size_band == band]
    f = map_track_lines(s, "", box=WORLD_BOX, colour=SIZE_BAND_COLOURS[band],
                        max_segments=8_000, width=1.1, opacity=0.5)
    for _n, _b in CORRIDORS.items():
        add_box(f, _b, _n, colour="#e8e2d4", width=0.8)
    panels.append((f"{band}  -  {s.mmsi.nunique():,} vessels", f))
fig = map_panels(panels, "World shipping routes by hull-length class, Nov 2025 to Feb 2026",
                 "AIS-matched vessels only; one panel per hull-length class. Corridor gates outlined.",
                 box=WORLD_BOX, ncols=2, legend=False)
save(fig, "fig03_world_routes_by_class", also_png=True)
fig.show()

232,130 world segments across 46,430 vessels (5s)


The classes have distinct geographies. Under 100 m is coastal and short-sea; the 300 m+ panel is
the long-haul trunk network (Cape, Malacca, the Red Sea corridor) with the Persian Gulf as one
spoke.

## 4.3 The deep-sea fleet before and after the change point

Vessels of 200 m and over constitute the Panamax-and-larger fleet that carries most seaborne
trade. Their global route map serves as the study's principal control: a change in the network as
a whole would be visible here.

In [13]:
rows = []
for label, periods, tag in WINDOWS:
    s = wseg[wseg.period.isin(periods) & (wseg.vessel_len >= 200)]
    days = window_days(glob_det, periods)
    fig = map_track_lines(
        s, f"World routes of vessels 200 m and over, {label}",
        f"{len(s):,} segments, {s.mmsi.nunique():,} vessels, {days} observed days.",
        box=WORLD_BOX, colour_by="vessel_group", height=700,
        max_segments=WORLD_ROUTE_SEGS, width=1.4, opacity=0.42)
    for _n, _b in CORRIDORS.items():
        add_box(fig, _b, _n, colour="#e8e2d4", width=1.0)
    save(fig, f"fig04_world_routes_ge200_{tag}", also_png=True)
    fig.show()
    allseg = wseg[wseg.period.isin(periods)]
    rows.append({"window": tag, "days": days,
                 "all_segments_per_day": round(len(allseg) / days, 1),
                 "ge200_segments_per_day": round(len(s) / days, 1),
                 "ge200_vessels": s.mmsi.nunique()})
display(pd.DataFrame(rows))

,window,days,all_segments_per_day,ge200_segments_per_day,ge200_vessels
0,pre,117,773.6,128.6,5605
1,post,118,758.9,125.2,5320


Global reconstructible traffic is essentially unchanged, with under 2% fewer segments per day
across all vessels and the deep-sea fleet flat, against a fall of roughly half inside the Hormuz
box (Section 6). The only visible difference between the panels is the spoke into the Persian
Gulf.

---
# 5. RQ1 and RQ2: Whether the Gulf change is measurable

The headline measurement is the share of detections with no AIS candidate. The principal
confound is the provider's change from pipeline v3 to v4 in March 2026, which coincides with the
month in which the series moves. The test is the identical statistic on four regions far from the
Gulf: a processing change would move every region together, whereas a behavioural change would
move only the affected one.

In [14]:
tc = treatment_control_table(glob_det)
display(tc.round(1))
print("Movement across the v3 -> v4 pipeline boundary (percentage points):")
display(pipeline_break_check(glob_det))
display(did_estimate(glob_det, PRE_MATCHED, POST_WINDOW))
cp = cusum_changepoint(dark_share_series(gulf))
print(f"Most likely change point : {cp['label']}  "
      f"({cp['mean_before']:.1f}% -> {cp['mean_after']:.1f}%, "
      f"R2 {cp['variance_explained']:.2f})")

,World,NE Atlantic,SE Asia,Gulf of Mexico,W Mediterranean,Gulf (treatment)
period,,,,,,
2025-09,21.4,7.3,30.9,26.3,11.3,16.7
2025-10,27.3,12.9,39.8,33.7,20.9,23.1
2025-11,26.9,11.3,35.7,33.5,20.6,23.0
2025-12,26.4,11.7,34.6,31.8,22.9,24.4
2026-01,25.9,13.8,31.0,29.8,16.0,24.4
2026-02,25.6,12.2,33.7,27.0,18.0,22.1
2026-03,25.3,12.1,33.6,21.0,19.2,41.2
2026-04,25.8,9.1,37.4,23.5,21.9,30.6
2026-05,30.7,12.9,50.8,21.3,22.0,45.0


Movement across the v3 -> v4 pipeline boundary (percentage points):


,2026-02 -> 2026-03 (pp)
World,-0.23
NE Atlantic,-0.10
SE Asia,-0.04
Gulf of Mexico,-6.03
W Mediterranean,1.29
Gulf (treatment),19.07


,treat_pre,treat_post,treat_delta,control_pre,control_post,control_delta,did_pp
0,23.5,40.08,16.58,26.19,27.12,0.94,15.64


Most likely change point : 2026-03  (22.3% -> 40.1%, R2 0.82)


In [15]:
long = tc.reset_index(names="period").melt(id_vars="period", var_name="series",
                                           value_name="dark_share")
ctrl_colours = {"Gulf (treatment)": CAT_COLOURS["no_ais_candidate"],
                "World": CAT_COLOURS["matched"]}
for _n in CONTROL_BOXES:
    ctrl_colours[_n] = "#4b5563"

fig = line_series(
    long, "period", "dark_share", "series",
    "Dark-detection share by region, Sep 2025 to Jun 2026",
    "Share of SAR detections with no AIS candidate. Control regions in grey.",
    ytitle="Dark detections (% of all detections)", xtitle="Month",
    colours=ctrl_colours, labels={k: k for k in ctrl_colours},
    direct_label=False, height=560)
add_pipeline_break(fig)
save(fig, "fig05_treatment_vs_control", also_png=True)
fig.show()

Across the pipeline boundary the world aggregate moves by −0.2 pp and three of the four controls
by under 1.5 pp; the Gulf of Mexico moves by −6.0 pp, in the opposite direction to the Gulf, which
moves by **+19.1 pp**. Over the matched windows the Gulf rises from 23.5% to 40.1% against a world
rise from 26.2% to 27.1%, a difference-in-differences estimate of **+15.6 pp**. A change-point scan
over the full monthly series places the break at March 2026 (means 22.3% before, 40.1% after,
R² 0.82). Endpoint months are avoided: September 2025 sits 6.7 points below every other pre-period
month, so quoting it against June 2026 would inflate the change by roughly nine points.

## 5.1 Observation-coverage correction

Raw monthly counts depend on the satellite's acquisition schedule. Expressing the same detections
per 1,000 km² of sea actually imaged removes that dependence. The two panels are presented
together because the result is that they have the same shape.

In [16]:
monthly = status_table(gulf)
raw_long = monthly.melt(id_vars="period", value_vars=list(AIS_STATUS),
                        var_name="ais_status", value_name="value")

adj = coverage_adjusted_series(det, cov, regions=GULF_REGIONS)
adj_long = adj.melt(id_vars="period",
                    value_vars=[f"{s}_per_1000km2" for s in AIS_STATUS],
                    var_name="ais_status", value_name="value")
adj_long["ais_status"] = adj_long["ais_status"].str.replace("_per_1000km2", "",
                                                            regex=False)
adj_long = adj_long.dropna(subset=["value"])

fig = paired_panels(
    raw_long, adj_long, "period", "value", "ais_status",
    "Gulf detections by AIS status, raw and per unit of sea imaged",
    "Gulf regions. The coverage-corrected series begins December 2025.",
    left_title="Raw monthly detections",
    right_title="Per 1,000 km2 of sea imaged",
    ytitle_left="Detections per month",
    ytitle_right="Detections per 1,000 km2",
    xtitle="Month", height=580)
save(fig, "fig06_levels_raw_vs_corrected", also_png=True)
fig.show()

---
# 6. RQ3: The strait, all vessels

## 6.1 Traffic volume through the strait

Two counting units are required. AIS-matched vessels can be counted by identity, as distinct
MMSIs. Dark detections have no identity and are counted as observations: the same dark vessel
imaged twice is two detections but one ship.

In [17]:
display(window_summary(det))
gate = gate_transits(det)
display(gate.round(1))

fig = line_series(
    gate.melt(id_vars="period", value_vars=["unique_mmsi", "dark_detections"],
              var_name="series", value_name="count"),
    "period", "count", "series",
    "Traffic through the Hormuz gate by month",
    "Distinct AIS-matched vessels and detections with no AIS candidate inside the gate.",
    ytitle="Count per month", xtitle="Month",
    colours={"unique_mmsi": CAT_COLOURS["matched"],
             "dark_detections": CAT_COLOURS["no_ais_candidate"]},
    labels={"unique_mmsi": "Distinct AIS-matched vessels",
            "dark_detections": "Dark detections"}, height=540)
add_pipeline_break(fig)
save(fig, "fig07_gate_transits", also_png=True)
fig.show()

,window,months,first,last,days_with_detections,detections,detections_per_day
0,pre,4,2025-11-01 01:18:54,2026-02-25 16:05:37,117,97853,836.4
1,post,4,2026-03-01 01:18:45,2026-06-29 15:28:06,117,91448,781.6


,period,detections,unique_mmsi,dark_detections,dark_share
0,2025-09,1708,836,539,31.6
1,2025-10,1753,747,771,44.0
2,2025-11,1594,735,682,42.8
3,2025-12,1708,726,756,44.3
4,2026-01,1916,758,907,47.3
5,2026-02,1485,680,631,42.5
6,2026-03,1213,361,687,56.6
7,2026-04,1346,485,475,35.3
8,2026-05,1066,394,550,51.6
9,2026-06,1292,421,737,57.0


## 6.2 Spatial distribution of detections

In [18]:
hz = det[in_box(det, HORMUZ_BOX)]
for label, periods, tag in WINDOWS:
    sub = hz[hz.period.isin(periods)]
    fig = map_detections(
        sub, f"Strait of Hormuz vessel detections, {label}",
        "Each point is one Sentinel-1 detection, coloured by AIS status.",
        box=HORMUZ_BOX, size=2.6, opacity=0.55, max_points=MAP_MAX_POINTS)
    add_ports(fig, HORMUZ_BOX)
    add_box(fig, HORMUZ_GATE, "gate")
    save(fig, f"fig08_map_{tag}", also_png=True)
    fig.show()

Resorting to unclean kill browser.


The blue lane through the strait in the first panel is the transit corridor. In the second it has
largely gone, and orange clusters occupy both sides of the strait: the anchorage inside the Gulf
off the UAE coast, and the Fujairah anchorage on the Gulf of Oman side.

## 6.3 Loitering

Loitering is defined as three or more detections of one MMSI remaining within 12 km over at least
36 hours below 1.5 knots, with no more than 120 hours between observations. Episodes are
apportioned across every month they overlap; attribution by start month produces a spurious
decline.

In [19]:
epi = find_loitering(gulf)
load = loitering_load(epi)
print(f"{len(epi):,} episodes across {epi.mmsi.nunique():,} vessels")
display(load.round(1))

fig = line_series(
    load.assign(series="vessel_days"), "period", "vessel_days", "series",
    "Loitering vessel-days in the Gulf by month",
    "Vessel-days of loitering, apportioned across the months each episode spans.",
    ytitle="Loitering vessel-days", xtitle="Month",
    colours={"vessel_days": CAT_COLOURS["no_ais_candidate"]},
    labels={"vessel_days": "Loitering vessel-days"}, height=520)
add_pipeline_break(fig)
save(fig, "fig09_loitering", also_png=True)
fig.show()

1,317 episodes across 755 vessels


,period,vessel_days,vessels,episodes_started
0,2025-09,1178.4,148,172
1,2025-10,652.2,115,100
2,2025-11,738.5,108,82
3,2025-12,1075.9,147,158
4,2026-01,1027.9,157,132
5,2026-02,615.2,87,66
6,2026-03,1272.2,190,218
7,2026-04,1988.2,253,243
8,2026-05,901.1,143,88
9,2026-06,411.8,57,58


Resorting to unclean kill browser.


---
# 7. RQ4: The strait by hull size

Section 6 treats a 24 m workboat and a 300 m tanker alike. This section repeats the same maps by
class.

## 7.1 Detections by class, before and after

In [20]:
panels = []
for band in SIZE_BAND_ORDER:
    for label, periods, tag in WINDOWS:
        s = hz[hz.period.isin(periods) & (hz.size_band == band)]
        f = map_detections(s, "", box=HORMUZ_BOX, colour_by="ais_status",
                           size=2.2, opacity=0.55, max_points=12_000)
        add_ports(f, HORMUZ_BOX, labels=False)
        add_box(f, HORMUZ_GATE, "gate")
        dark = 100 * (s.ais_status == "no_ais_candidate").mean()
        panels.append((f"{SHORT_BAND[band]}, {label}  -  {len(s):,} det., {dark:.0f}% dark", f))
fig = map_panels(panels, "Strait of Hormuz detections by hull-length class and window",
                 "Rows are hull-length classes, columns the two windows. Coloured by AIS status.",
                 box=HORMUZ_BOX, ncols=2, width=1100, vspace=0.045, title_band=190)
save(fig, "fig10_hormuz_by_class_grid", also_png=True)
fig.show()

Every class begins with the same blue lane through the strait and ends with it thinned and
orange. The dark share after the change point rises with size: 42% under 100 m, 47% above 300 m.

## 7.2 Spatial change by vessel size

Differencing the two windows on the coverage grid shows where vessels accumulated and where they
disappeared. The measure is a normalised difference, `(post − pre) / (post + pre)`, so that a
halving and a doubling read identically in every panel regardless of the number of ships in the
class; cells with fewer than four detections in total are omitted.

In [21]:
def window_rates(sub):
    """Detections per observed day per cell, per window, from summed counts."""
    grid = congestion_grid(sub, cov)
    out = {}
    for _, periods, tag in WINDOWS:
        g = (grid[grid.period.isin(periods)]
             .groupby(["ix", "iy"], observed=True)
             .agg(detections=("detections", "sum"), obs_days=("obs_days", "sum"),
                  lon=("lon", "first"), lat=("lat", "first")))
        g["rate"] = g.detections / g.obs_days
        out[tag] = g
    j = out["pre"][["rate", "detections", "lon", "lat"]].rename(
        columns={"rate": "pre", "detections": "n_pre"}).join(
        out["post"][["rate", "detections"]].rename(
            columns={"rate": "post", "detections": "n_post"}), how="inner")
    j["delta"] = j.post - j.pre
    j["rel"] = (j.post - j.pre) / (j.post + j.pre)
    return j.reset_index()


THRESHOLDS = [(0, "All vessels"), (100, "100 m and over"),
              (200, "200 m and over"), (300, "300 m and over")]
panels = []
for thr, lab in THRESHOLDS:
    chg = window_rates(hz[hz.length_m >= thr])
    chg = chg[(chg.n_pre + chg.n_post) >= 4]
    f = map_change(chg, "", box=HORMUZ_BOX, value="rel", limit=1.0,
                   colourbar_title="Relative<br>change")
    add_ports(f, HORMUZ_BOX, labels=False)
    add_box(f, HORMUZ_GATE, "gate")
    panels.append((lab, f))
fig = map_panels(panels, "Relative change in detection density by minimum vessel length",
                 "Detections per observed day, Mar to Jun 2026 against Nov 2025 to Feb 2026. "
                 "Cells with under four detections omitted.",
                 box=HORMUZ_BOX, ncols=2, legend=False)
fig.data[0].marker.colorbar.update(tickvals=[-1, -1/3, 0, 1/3, 1],
                                   ticktext=["emptied", "halved", "same", "doubled", "new"])
save(fig, "fig11_change_ladder", also_png=True)
fig.show()

On all vessels the pattern is diffuse. Each step up the threshold sharpens it, until at 200 m and
over two features dominate: the transit lane through the strait is blue, and the anchorages on
both sides of it are red.

## 7.3 Routes of vessels 200 m and over

In [22]:
seg = add_vessel_band(build_segments(det), det)
near = seg[in_box(seg, HORMUZ_BOX)]

for label, periods, tag in WINDOWS:
    s = near[near.period.isin(periods) & (near.vessel_len >= 200)]
    fig = map_track_lines(
        s, f"Routes of vessels 200 m and over, {label}",
        f"{len(s):,} segments, {s.mmsi.nunique():,} vessels over "
        f"{window_days(det, periods)} observed days.",
        box=HORMUZ_BOX, colour_by="vessel_group", max_segments=ROUTE_MAX_SEGS,
        width=1.9, opacity=0.5)
    add_ports(fig, HORMUZ_BOX, labels=False)
    add_box(fig, HORMUZ_GATE, "gate")
    save(fig, f"fig12_routes_ge200_{tag}", also_png=True)
    fig.show()

tab = (near[near.period.isin(PRE_MATCHED + POST_WINDOW)]
       .assign(window=lambda d: np.where(d.period.isin(PRE_MATCHED), "pre", "post"))
       .groupby(["size_band", "window"], observed=True)
       .agg(segments=("mmsi", "size"), vessels=("mmsi", "nunique"))
       .unstack().reindex(SIZE_BAND_ORDER))
display(tab)

segments       vessels      
window                               post   pre    post   pre
size_band                                                    
Under 100 m                          1089  2218     653  1041
100-200 m (feeder / handysize)       1025  1787     515   988
200-300 m (Panamax class)             566   859     229   531
300 m+ (post-Panamax and larger)      125   202      41   142

Before the change point a dense corridor runs through the strait into the Gulf. After it, the
remaining segments are short movements inside the Gulf, consistent with vessels waiting rather
than transiting. GFW has no tanker class, so tankers fall into the unclassified group, which is
why that group dominates at this size.

## 7.4 Departure or concealment: radar counts by class

Route maps cannot distinguish departure from concealment, because a vessel that stops
broadcasting disappears from them in either case. The following counts every hull the radar
observed, separately from the hulls that matched AIS and those that did not, by class.

In [23]:
def pct_change_by_band(df, days_pre, days_post):
    d = df[df.period.isin(PRE_MATCHED + POST_WINDOW)].assign(
        window=lambda d: np.where(d.period.isin(PRE_MATCHED), "pre", "post"))
    rows = []
    for status, lab in [(None, "All hulls detected by radar"),
                        ("matched", "AIS-matched (identifiable)"),
                        ("no_ais_candidate", "No AIS candidate (dark)")]:
        s = d if status is None else d[d.ais_status == status]
        t = s.groupby(["window", "size_band"], observed=True).size().unstack(0)
        t = t.reindex(SIZE_BAND_ORDER)
        chg = 100 * (t.post / days_post) / (t.pre / days_pre) - 100
        for band, v in chg.items():
            rows.append({"band": SHORT_BAND[band], "series": lab, "pct": v})
    return pd.DataFrame(rows)


gulf_mask = in_box(glob_det, GULF_BOX)
gulf_g, rest = glob_det[gulf_mask], glob_det[~gulf_mask]
bars_gulf = pct_change_by_band(gulf_g, window_days(gulf_g, PRE_MATCHED),
                               window_days(gulf_g, POST_WINDOW))
bars_rest = pct_change_by_band(rest, window_days(rest, PRE_MATCHED),
                               window_days(rest, POST_WINDOW))
print("Gulf, % change per observed day:")
display(bars_gulf.pivot(index="band", columns="series", values="pct")
        .reindex([SHORT_BAND[b] for b in SIZE_BAND_ORDER]).round(1))
print("Rest of world:")
display(bars_rest.pivot(index="band", columns="series", values="pct")
        .reindex([SHORT_BAND[b] for b in SIZE_BAND_ORDER]).round(1))
display(size_gradient(gulf))

Gulf, % change per observed day:


series,AIS-matched (identifiable),All hulls detected by radar,No AIS candidate (dark)
band,,,
Under 100 m,-36.5,-13.9,41.0
100-200 m,-31.9,0.7,67.0
200-300 m,-39.1,0.8,89.5
300 m+,-40.6,9.1,127.8


Rest of world:


series,AIS-matched (identifiable),All hulls detected by radar,No AIS candidate (dark)
band,,,
Under 100 m,-9.9,-6.4,-6.4
100-200 m,-3.7,-2.5,4.0
200-300 m,2.9,2.7,-5.5
300 m+,3.1,2.5,-7.0


,size_band,detections,pct_of_all,dark_pre_pct,dark_post_pct,swing_pp
0,Under 100 m,76078,55.6,23.0,37.7,14.7
1,100-200 m (feeder / handysize),39585,29.0,25.3,42.0,16.7
2,200-300 m (Panamax class),16779,12.3,22.8,42.9,20.1
3,300 m+ (post-Panamax and larger),4275,3.1,21.7,45.2,23.5


In [24]:
from plotly.subplots import make_subplots

SERIES_COL = {"All hulls detected by radar": INK,
              "AIS-matched (identifiable)": CAT_COLOURS["matched"],
              "No AIS candidate (dark)": CAT_COLOURS["no_ais_candidate"]}
order = [SHORT_BAND[b] for b in SIZE_BAND_ORDER]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                    subplot_titles=("Persian Gulf, Hormuz and Gulf of Oman",
                                    "Rest of the world (control)"))
for col, frame in [(1, bars_gulf), (2, bars_rest)]:
    for series, colour in SERIES_COL.items():
        vals = frame[frame.series == series].set_index("band").loc[order, "pct"]
        fig.add_trace(go.Bar(
            x=[o.replace(" ", "<br>", 1) for o in order], y=vals, name=series,
            marker_color=colour, legendgroup=series, showlegend=(col == 1),
            text=[f"{v:+.0f}%" for v in vals], textposition="outside",
            textfont=dict(size=11, color=INK),
            hovertemplate=f"{series}: %{{y:+.1f}}%<extra></extra>"), row=1, col=col)
fig.update_layout(**_base_layout(
    "Change in detections per observed day by hull-length class and AIS status",
    "Mar to Jun 2026 against Nov 2025 to Feb 2026, Gulf and rest of world.", height=560))
fig.update_layout(barmode="group", bargap=0.25, bargroupgap=0.05, margin_t=170,
                  legend=dict(y=1.10, yanchor="bottom"))
fig.update_yaxes(title_text="Change (%)", gridcolor=GRID, zeroline=True,
                 zerolinecolor=INK_MUTED, zerolinewidth=1, linecolor=GRID,
                 tickfont=dict(color=INK_MUTED), title_font=dict(color=INK_MUTED, size=12),
                 range=[-60, 150])
fig.update_xaxes(gridcolor=GRID, linecolor=GRID, tickfont=dict(color=INK_MUTED))
for ann in fig.layout.annotations:
    ann.font.color = INK
    ann.font.size = 12
save(fig, "fig13_stayed_not_identifying", also_png=True)
fig.show()

Resorting to unclean kill browser.


**Radar observes the same number of large hulls in the Gulf after the change point as before**:
100 to 300 m flat and 300 m+ up 9%, while AIS-matched detections of those classes fall by a third
to two fifths and dark detections rise by 67%, 89% and 128%. Outside the Gulf every cell of the
equivalent table lies between −10% and +4%. The large vessels did not leave; they remained, and
between a third and two fifths of them ceased to match AIS.

The dark share within each class confirms the ordering: every class begins between 21.7% and
25.3%, and the increase grows monotonically with size, from +14.7 points under 100 m to +23.5
above 300 m. Detection noise acts most strongly on the smallest hulls and would produce the
opposite gradient.

---
# 8. RQ5: The identifiable fleet

Section 7 shows that large hulls remained; Section 6.1 shows that identified transits halved.
The two are reconciled through the global identifier: for every vessel seen in the Gulf before
the change point, its subsequent appearance is classified as in the Gulf, elsewhere in the world,
or nowhere. "Nowhere" includes going dark but also lay-up, sale and simply not being imaged, so
control regions are reported alongside.

In [25]:
fate = fleet_fate_table(glob_det)
display(fate[["region", "vessel_type", "vessels",
              "stayed_pct", "elsewhere_pct", "vanished_pct"]])
print("Cargo, treatment against control mean:")
display(relocation_contrast(fate, "cargo"))

OUTCOME = {"stayed_pct": ("Still in region", "#3987e5"),
           "elsewhere_pct": ("Elsewhere in the world", "#199e70"),
           "vanished_pct": ("Not detected anywhere", "#d95926")}
sub = fate[fate.vessel_type.isin(["cargo", "other", "fishing"])].copy()
sub["label"] = sub["vessel_type"] + ", " + sub["region"]
sub = sub.sort_values(["vessel_type", "region"])

fig = go.Figure()
for col, (lab, colr) in OUTCOME.items():
    fig.add_trace(go.Bar(
        y=sub["label"], x=sub[col], orientation="h", name=lab,
        marker=dict(color=colr, line=dict(width=2, color=SURFACE)),
        hovertemplate="%{y}<br>" + lab + ": %{x:.1f}%<extra></extra>"))
fig.update_layout(**_base_layout(
    "Subsequent location of vessels observed in each region before the change point",
    "Share of pre-window vessels by subsequent location; control regions shown for "
    "comparison.", height=620))
fig.update_layout(barmode="stack")
style_axes(fig, "Share of pre-window vessels (%)", None)
save(fig, "fig14_fleet_fate", also_png=True)
fig.show()

,region,vessel_type,vessels,stayed_pct,elsewhere_pct,vanished_pct
0,Gulf (treatment),other,5414,52.6,26.1,21.2
1,Gulf (treatment),cargo,2033,34.0,58.0,8.0
2,Gulf (treatment),fishing,904,41.6,0.9,57.5
3,Gulf (treatment),noisy_vessel,234,43.6,8.5,47.9
4,Gulf (treatment),gear,222,36.0,0.0,64.0
5,Gulf (treatment),passenger,178,37.6,5.6,56.7
6,SE Asia,other,14946,65.0,12.6,22.4
7,SE Asia,cargo,9866,62.9,31.9,5.2
8,SE Asia,fishing,1949,43.4,7.3,49.3
9,SE Asia,gear,1515,23.4,1.5,75.1


Cargo, treatment against control mean:


,outcome,gulf,control_mean,gap_pp
0,stayed,34.0,60.0,-26.0
1,elsewhere,58.0,35.7,22.3
2,vanished,8.0,4.3,3.7


Only 34.0% of Gulf cargo vessels are seen in the Gulf again, against a control mean of 60.0%, and
almost the entire 26-point shortfall reappears elsewhere (58.0% against 35.7%); the excess vanish
rate is +3.7 points. Read with Section 7, the compliant fleet present before the break left and
continued broadcasting, and a non-broadcasting fleet of the same hull sizes replaced it.

## 8.1 Port-level displacement

Port activity is measured as detections in a 5 to 40 km approach annulus, the water a vessel
occupies while waiting, manoeuvring or berthing. The inner 5 km is excluded because returns over
the berth are dominated by cranes and moored lighters.

In [26]:
import numpy as np
import pandas as pd


# Ports reachable only by transiting the Strait of Hormuz.
INSIDE_GULF = {
    "Jebel Ali (AE)", "Khalifa/Abu Dhabi (AE)", "Ruwais (AE)", "Ras Tanura (SA)",
    "Jubail (SA)", "Kharg Island (IR)", "Bandar Abbas (IR)", "Umm Qasr (IQ)",
    "Basra Oil Terminal(IQ)", "Kuwait/Al Ahmadi (KW)", "Hamad (QA)",
    "Ras Laffan (QA)", "Mina Salman (BH)",
}
# Ports on the seaward side that serve overlapping trade without a transit.
OUTSIDE_BYPASS = {
    "Fujairah (AE)", "Sohar (OM)", "Duqm (OM)", "Salalah (OM)",
    "Karachi (PK)", "Mundra (IN)", "Jawaharlal Nehru (IN)",
}


def port_frame():
    rows = []
    for name, (lon, lat) in C.PORTS.items():
        side = ("inside_gulf" if name in INSIDE_GULF
                else "bypass" if name in OUTSIDE_BYPASS else "other")
        rows.append({"port": name, "lon": lon, "lat": lat, "side": side})
    return pd.DataFrame(rows)


def assign_nearest_port(df, max_km=40.0):
    """Attach the nearest port and its distance, within ``max_km``."""
    ports = port_frame()
    plat = ports["lat"].to_numpy()
    plon = ports["lon"].to_numpy()

    lat = df["lat"].to_numpy(dtype="float64")
    lon = df["lon"].to_numpy(dtype="float64")
    nearest = np.empty(len(df), dtype="int64")
    dist = np.empty(len(df), dtype="float32")

    step = 20000
    for i in range(0, len(df), step):
        d = tracks.haversine_km(lat[i:i + step, None], lon[i:i + step, None],
                                plat[None, :], plon[None, :])
        nearest[i:i + step] = d.argmin(axis=1)
        dist[i:i + step] = d.min(axis=1)

    out = df.copy()
    out["port"] = pd.Categorical(ports["port"].to_numpy()[nearest])
    out["port_side"] = pd.Categorical(ports["side"].to_numpy()[nearest])
    out["port_dist_km"] = dist
    out.loc[out["port_dist_km"] > max_km, ["port", "port_side"]] = np.nan
    return out


def port_activity(df, freq="M", inner_km=5.0, outer_km=40.0):
    """Detections in each port's approach annulus, per period.

    The inner radius is excluded because detections on top of the berth are
    dominated by quay cranes, moored barges and shore clutter rather than
    arriving traffic.
    """
    d = assign_nearest_port(df, max_km=outer_km)
    d = d[d["port"].notna() & (d["port_dist_km"] >= inner_km)]
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)

    out = (d.groupby(["period", "port", "port_side"], observed=True)
           .agg(detections=("lat", "size"),
                dark=("ais_status", lambda s: (s == "no_ais_candidate").sum()),
                unique_mmsi=("mmsi", lambda s: s.dropna().nunique()))
           .reset_index())
    out["dark_share"] = 100 * out["dark"] / out["detections"]
    return out


def side_shares(activity):
    """Share of port-approach detections inside the Gulf vs the bypass coast."""
    piv = (activity.pivot_table(index="period", columns="port_side",
                                values="detections", aggfunc="sum", observed=True)
           .fillna(0))
    for col in ("inside_gulf", "bypass", "other"):
        if col not in piv:
            piv[col] = 0.0
    piv["total"] = piv[["inside_gulf", "bypass", "other"]].sum(axis=1)
    piv["inside_share"] = 100 * piv["inside_gulf"] / piv["total"]
    piv["bypass_share"] = 100 * piv["bypass"] / piv["total"]
    # The headline ratio: bypass activity per unit of inside-Gulf activity.
    piv["bypass_ratio"] = piv["bypass"] / piv["inside_gulf"].replace(0, np.nan)
    return piv.reset_index()


def port_change(activity, pre_periods, post_periods):
    """Per-port change in approach detections between two windows."""
    piv = (activity.pivot_table(index="port", columns="period",
                                values="detections", aggfunc="sum", observed=True)
           .fillna(0))
    pre = piv[[p for p in pre_periods if p in piv.columns]].mean(axis=1)
    post = piv[[p for p in post_periods if p in piv.columns]].mean(axis=1)
    sides = port_frame().set_index("port")["side"]

    out = pd.DataFrame({"pre": pre, "post": post})
    out["side"] = sides.reindex(out.index)
    out["abs_change"] = out["post"] - out["pre"]
    out["pct_change"] = 100 * out["abs_change"] / out["pre"].replace(0, np.nan)
    return out.sort_values("pct_change", ascending=False).reset_index()


def anchorage_queues(loitering_episodes, max_km=60.0):
    """Attribute loitering episodes to the port they are waiting off."""
    if loitering_episodes.empty:
        return loitering_episodes
    epi = assign_nearest_port(loitering_episodes, max_km=max_km)
    epi = epi[epi["port"].notna()]
    return (epi.groupby(["period", "port", "port_side"], observed=True)
            .agg(episodes=("mmsi", "size"),
                 vessels=("mmsi", "nunique"),
                 median_span_h=("span_h", "median"))
            .reset_index())

In [27]:
act = port_activity(det)
ss = side_shares(act)
display(ss[["period", "inside_gulf", "bypass", "bypass_ratio"]].round(2))
ss_big = side_shares(port_activity(commercial(det)))
print("Vessels 100 m and over:")
display(ss_big[["period", "inside_gulf", "bypass", "bypass_ratio"]].round(2))

pc = port_change(act, PRE_MATCHED, POST_WINDOW)
display(pc.round(1))

pc2 = pc.dropna(subset=["pct_change"]).sort_values("pct_change")
fig = go.Figure(go.Bar(
    x=pc2["pct_change"], y=pc2["port"], orientation="h",
    marker=dict(color=[CAT_COLOURS["no_ais_candidate"] if s == "bypass"
                       else CAT_COLOURS["matched"] if s == "inside_gulf"
                       else "#4b5563" for s in pc2["side"]], line=dict(width=0)),
    hovertemplate="%{y}: %{x:.0f}%<extra></extra>"))
fig.update_layout(**_base_layout(
    "Change in port-approach detections by port",
    "Mar to Jun 2026 against Nov 2025 to Feb 2026. Orange: bypass coast; blue: inside the Gulf.",
    height=660, legend=False))
style_axes(fig, "Change (%)", None)
save(fig, "fig15_port_change", also_png=True)
fig.show()

port_side,period,inside_gulf,bypass,bypass_ratio
0,2025-09,3376,1762,0.52
1,2025-10,3283,1815,0.55
2,2025-11,2708,1858,0.69
3,2025-12,3585,1888,0.53
4,2026-01,3454,2205,0.64
5,2026-02,2664,1537,0.58
6,2026-03,3143,2498,0.79
7,2026-04,3073,1975,0.64
8,2026-05,2485,2207,0.89
9,2026-06,2418,2670,1.10


Vessels 100 m and over:


port_side,period,inside_gulf,bypass,bypass_ratio
0,2025-09,1572,1308,0.83
1,2025-10,1475,1336,0.91
2,2025-11,1324,1262,0.95
3,2025-12,1701,1356,0.80
4,2026-01,1694,1544,0.91
5,2026-02,1393,1050,0.75
6,2026-03,1527,1715,1.12
7,2026-04,1169,1394,1.19
8,2026-05,913,1584,1.73
9,2026-06,908,2042,2.25


,port,pre,post,side,abs_change,pct_change
0,Duqm (OM),30.0,63.2,bypass,33.2,110.8
1,Sohar (OM),279.0,539.2,bypass,260.2,93.3
2,Khalifa/Abu Dhabi (AE),119.5,194.5,inside_gulf,75.0,62.8
3,Salalah (OM),27.0,40.0,bypass,13.0,48.1
4,Mundra (IN),235.5,337.2,bypass,101.8,43.2
5,Umm Qasr (IQ),7.8,10.8,inside_gulf,3.0,38.7
6,Mina Salman (BH),76.0,101.8,inside_gulf,25.8,33.9
7,Jawaharlal Nehru (IN),122.5,155.2,bypass,32.8,26.7
8,Ras Laffan (QA),372.5,417.5,inside_gulf,45.0,12.1
9,Kuwait/Al Ahmadi (KW),109.2,116.8,inside_gulf,7.5,6.9


## 8.2 Corridor-level displacement

Detections are counted through seven chokepoint gates as a share of world detections, then by
size class, because the result on all vessels is not supported by the deep-sea fleet.

In [28]:
import numpy as np
import pandas as pd



def corridor_counts(glob_det, freq="M", corridors=None):
    """Detections per month inside each chokepoint gate, plus global totals."""
    corridors = corridors or C.CORRIDORS
    d = glob_det.copy()
    d["period"] = d["timestamp"].dt.to_period(freq).astype(str)

    world = d.groupby("period", observed=True).size().rename("world_detections")

    rows = []
    for name, box in corridors.items():
        sub = d[build.in_box(d, box)]
        if sub.empty:
            continue
        g = sub.groupby("period", observed=True).agg(
            detections=("lat", "size"),
            dark=("ais_status", lambda s: (s == "no_ais_candidate").sum()),
            unique_mmsi=("mmsi", lambda s: s.dropna().nunique()),
        )
        g["corridor"] = name
        rows.append(g.reset_index())

    out = pd.concat(rows, ignore_index=True)
    out = out.merge(world.reset_index(), on="period", how="left")
    # Share of world detections: divides out the global acquisition volume.
    out["per_10k_world"] = 1e4 * out["detections"] / out["world_detections"]
    out["dark_share"] = 100 * out["dark"] / out["detections"]
    return out.sort_values(["corridor", "period"], ignore_index=True)


def corridor_index(counts, base_periods=None):
    """Each corridor indexed to 100 at its own baseline mean.

    Corridors differ greatly in absolute traffic, so absolute counts on one
    axis would be unreadable. Indexing to each
    corridor's own pre-period mean puts them on a comparable scale where the
    question is "did this route grow or shrink", not "which is busiest".
    """
    base_periods = base_periods or C.PRE_MATCHED
    out = []
    for name, grp in counts.groupby("corridor", observed=True):
        grp = grp.sort_values("period").copy()
        base = grp.loc[grp["period"].isin(base_periods), "per_10k_world"].mean()
        grp["index"] = 100 * grp["per_10k_world"] / base if base else np.nan
        out.append(grp)
    return pd.concat(out, ignore_index=True)


def route_shift(counts, pre=None, post=None, pairs=None):
    """Change in each corridor between two windows, and the paired ratios.

    The paired ratio is the informative quantity: if Red Sea traffic falls
    while Cape traffic rises, the ratio between them moves far more decisively
    than either series alone.
    """
    pre = pre or C.PRE_MATCHED
    post = post or C.POST_WINDOW
    pairs = pairs or C.ROUTE_PAIRS

    piv = counts.pivot_table(index="corridor", columns="period",
                             values="per_10k_world", aggfunc="sum", observed=True)
    pre_cols = [p for p in pre if p in piv.columns]
    post_cols = [p for p in post if p in piv.columns]

    tab = pd.DataFrame({
        "pre": piv[pre_cols].mean(axis=1),
        "post": piv[post_cols].mean(axis=1),
    })
    tab["pct_change"] = 100 * (tab["post"] - tab["pre"]) / tab["pre"].replace(0, np.nan)
    tab = tab.sort_values("pct_change", ascending=False).reset_index()

    ratios = []
    idx = tab.set_index("corridor")
    for label, (a, b) in pairs.items():
        if a in idx.index and b in idx.index:
            ratios.append({
                "pair": label,
                "pre_ratio": idx.loc[b, "pre"] / idx.loc[a, "pre"],
                "post_ratio": idx.loc[b, "post"] / idx.loc[a, "post"],
            })
    ratio_tab = pd.DataFrame(ratios)
    if not ratio_tab.empty:
        ratio_tab["change_pct"] = 100 * (
            ratio_tab["post_ratio"] - ratio_tab["pre_ratio"]
        ) / ratio_tab["pre_ratio"]
    return tab, ratio_tab

In [29]:
counts_c = corridor_counts(glob_det)
idx = corridor_index(counts_c)
corridor_colours = dict(zip(CORRIDORS, [CAT_COLOURS["no_ais_candidate"], "#eda100",
                                        "#e34948", CAT_COLOURS["matched"], "#199e70",
                                        "#9085e9", "#6b7280"]))
fig = line_series(
    idx, "period", "index", "corridor",
    "Chokepoint traffic as a share of world detections, indexed to the pre-window mean",
    "Index = 100 at each corridor's Nov 2025 to Feb 2026 mean.",
    ytitle="Index (pre-period mean = 100)", xtitle="Month",
    colours=corridor_colours, labels={n: n for n in CORRIDORS},
    direct_label=False, height=580)
add_pipeline_break(fig)
save(fig, "fig16_corridor_index", also_png=True)
fig.show()

rows = []
for band in SIZE_BAND_ORDER:
    g = glob_det[glob_det.size_band == band]
    for corr, box in CORRIDORS.items():
        a = in_box(g[g.period.isin(PRE_MATCHED)], box).mean() * 100
        b = in_box(g[g.period.isin(POST_WINDOW)], box).mean() * 100
        rows.append({"corridor": corr, "band": SHORT_BAND[band],
                     "change_%": 100 * (b / a - 1) if a else np.nan})
corr_by_band = (pd.DataFrame(rows).pivot(index="corridor", columns="band", values="change_%")
                [[SHORT_BAND[b] for b in SIZE_BAND_ORDER]])
print("Change in corridor share of world detections, by size class (%):")
display(corr_by_band.round(1))

Change in corridor share of world detections, by size class (%):


band,Under 100 m,100-200 m,200-300 m,300 m+
corridor,,,,
Bab el-Mandeb,51.1,-5.5,18.1,-3.9
Cape of Good Hope,-7.7,8.0,2.3,5.8
Gulf of Guinea,2.5,17.9,15.5,13.0
Mozambique Channel,98.4,-5.9,-12.8,-26.8
Strait of Hormuz,-27.0,-13.8,-35.0,-49.0
Strait of Malacca,-0.5,-19.9,-28.5,-26.9
Suez / Gulf of Suez,-52.1,-3.5,10.3,36.0


The Hormuz gate falls in every class. No other corridor moves consistently: on all vessels the
Mozambique Channel nearly doubles and Suez halves, but both movements are confined to the
under-100 m class. For hulls of 100 m and over Suez is flat to up and the Cape within a few
percent of unchanged. The apparent increase in traffic around Africa is small-vessel traffic.
Displacement is regional and port-level, not global.

---
# 9. Conclusions

1. **Dark shipping in the Gulf rose sharply**, from a pre-window mean of 22.3% of detections to
   40.1% after the change point, against a world baseline near 26%. Difference-in-differences
   estimate **+15.6 percentage points** over the matched windows; change point March 2026.

2. **The change is local and not instrumental.** The world aggregate and three of four control
   regions move under 1.5 points across the provider's pipeline change, the fourth moves 6 points
   in the opposite direction, and the Gulf moves +19.1. The result survives normalisation by the
   sea area actually imaged.

3. **The large ships remained.** Radar detections of hulls 100 m and over are flat to up 9% per
   observed day; AIS-matched detections of the same classes fall by a third to two fifths and
   dark detections rise 67 to 128%. Outside the Gulf every equivalent cell is within ±10%.

4. **The effect scales with size.** The dark-share increase runs from +14.7 points under 100 m to
   +23.5 points above 300 m; detection noise would produce the opposite ordering.

5. **The corridor emptied and the anchorages filled.** Identified transits halved (836 to 421
   distinct vessels per month), loitering tripled to an April peak, and at 200 m and over the
   change map shows an emptied lane between filled anchorages on both sides of the strait.

6. **The identifiable cargo fleet relocated**: 34% seen in the Gulf again against a 60% control
   mean, with the shortfall reappearing elsewhere, and a non-broadcasting fleet of the same hull
   sizes replaced it.

7. **Displacement is regional, not global.** Bypass-coast port activity nearly triples relative
   to inside-Gulf activity for the commercial fleet; the apparent re-routing around Africa is
   confined to vessels under 100 m.

## Limitations

The analysis establishes that a change occurred, that it was local, and that it was not
instrumental, not that the conflict caused it. A dark detection means no AIS candidate in the
provider's matching, not demonstrated intent. Radar length is not vessel type, and a class
boundary is blurred by sea state. Loitering rests on a revisit interval of roughly seven days.
Port activity is an approach-annulus proxy. Coverage correction is unavailable before December
2025 and outside the study area, and no same-season 2025 baseline was available to exclude
seasonality directly; the world control is the substitute.

In [30]:
print(f"AIS acceptance threshold      : {MATCH_SCORE_THRESHOLD}")
print(f"Coverage grid resolution      : {GRID_RES} degrees")
print(f"Size classes                  : {[b[2] for b in SIZE_BANDS]}")
print(f"Pre window / post window      : {PRE_MATCHED} / {POST_WINDOW}")
print(f"Observed days per window      : {window_days(det, PRE_MATCHED)} / "
      f"{window_days(det, POST_WINDOW)}")
print(f"\nFigures written: {len(sorted(FIG_DIR.glob('*.html')))} to {FIG_DIR}")

AIS acceptance threshold      : 0.01
Coverage grid resolution      : 0.1 degrees
Size classes                  : ['Under 100 m', '100-200 m (feeder / handysize)', '200-300 m (Panamax class)', '300 m+ (post-Panamax and larger)']
Pre window / post window      : ['2025-11', '2025-12', '2026-01', '2026-02'] / ['2026-03', '2026-04', '2026-05', '2026-06']
Observed days per window      : 117 / 117

Figures written: 18 to c:\Users\tatee\Desktop\Data Vis\CW2\DSM050_Final_v2\figures
